# 🧭 Ruta Ibérica: asesor de viajes con IA para Barcelona, Madrid y Mallorca
**Primer Parcial de NLP · Desafío LangChain + Chainlit**

**Integrantes:** Guillermo Prieto · Marco Fernandez · Javier Alonso Desimone · Rodrigo Sanchez Lopez · Ignacio Mititieri

## Problema
Planificar un viaje por varias ciudades implica revisar cientos de alojamientos y miles de reseñas en distintos idiomas.
**Ruta Ibérica** es un agente conversacional que, para cada destino del viaje, filtra alojamientos **disponibles** y con
**capacidad para todo el grupo**, **lee las reseñas por el viajero** y presenta **4 opciones**:
🧭 recomendada por nuestro modelo · ⭐ mejor rateada · 💸 más barata · 💎 más cara, cada una con un **puntaje de 1 a 10**.

## Datos
[Inside Airbnb](https://insideairbnb.com/get-the-data/) (fuente pública y rastreable), relevamientos de junio de 2026:
`listings` (precio, capacidad, ubicación, ratings, foto), `calendar` (disponibilidad diaria por 12 meses) y
`reviews` (texto de las reseñas, en más de 6 idiomas). Complemento: clima de [Open-Meteo](https://open-meteo.com/) (API gratuita).

## Arquitectura
```
CSV Inside Airbnb ─► data_prep ─► catálogo curado (1.500 alojamientos) + calendario + reseñas
                                        │
      LCEL: prompt | LLM.with_structured_output(Pydantic)                 ◄── RunnableParallel (LLM + léxico + estadísticas)
      .batch() sobre el catálogo ─► DataFrame enriquecido (reviews_enriched.parquet)
                                        │
      Modelo de puntaje 1-10 (calidad bayesiana · reseñas IA · precio/calidad · ubicación · capacidad)
                                        │
      Agente create_agent + 8 tools + memoria por sesión (checkpointer, thread_id) + métricas (callback)
                                        │
      Interfaz Chainlit: streaming, tarjetas con fotos, mapas, radar, clima, sliders de pesos, métricas
```
LLM: **Groq free tier** (`openai/gpt-oss-120b` para el agente, `openai/gpt-oss-20b` / `qwen/qwen3.8-27b` para el batch):
el uso real cuesta US$ 0 y el costo se **simula** con precios de lista.

## 0 · Configuración del entorno

In [ ]:
import os, sys
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    %pip install -q "langchain>=1.4" "langchain-core>=1.6" "langchain-groq>=1.1" "langgraph>=1.0" "chainlit>=2.12" \
        "plotly==5.24.1" pyarrow python-dotenv requests
    os.makedirs("/content/ruta_iberica", exist_ok=True)
    %cd /content/ruta_iberica
for carpeta in ["core", "data/raw", "data/processed", "public/elements", ".chainlit"]:
    os.makedirs(carpeta, exist_ok=True)
print("Directorio de trabajo:", os.getcwd())

In [ ]:
# La API key NUNCA se escribe en la notebook: se toma del entorno, de los Secrets de Colab o se pide con getpass.
from getpass import getpass
if not os.getenv("GROQ_API_KEY"):
    try:
        from google.colab import userdata
        os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")
    except Exception:
        from dotenv import load_dotenv
        load_dotenv()
if not os.getenv("GROQ_API_KEY"):
    os.environ["GROQ_API_KEY"] = getpass("GROQ_API_KEY (https://console.groq.com/keys): ")
print("API key cargada ✔")

## 1 · Módulos del proyecto
Cada celda escribe un archivo del paquete `core/` (la app de Chainlit importa exactamente este código).

In [ ]:
%%writefile core/__init__.py
"""Paquete de Ruta Ibérica: datos, NLP, puntaje, tools y agente."""

In [ ]:
%%writefile core/config.py
"""Configuración central: rutas, modelos y precios de referencia para la simulación económica."""
import os
from getpass import getpass
from pathlib import Path

from dotenv import load_dotenv

ROOT = Path(__file__).resolve().parents[1]
PROCESSED = ROOT / "data" / "processed"
load_dotenv(ROOT / ".env")

# Modelos gratuitos de Groq. Cada modelo tiene su propia cuota diaria en el free tier,
# por eso el agente y el batch usan modelos distintos.
MODELO_AGENTE = os.getenv("MODELO_AGENTE", "openai/gpt-oss-120b")
MODELO_BATCH = os.getenv("MODELO_BATCH", "openai/gpt-oss-20b")

DESTINOS = {
    "barcelona": {"nombre": "Barcelona", "centro": (41.3874, 2.1686), "emoji": "🏛️"},
    "madrid": {"nombre": "Madrid", "centro": (40.4168, -3.7038), "emoji": "🎨"},
    "mallorca": {"nombre": "Mallorca", "centro": (39.6953, 3.0176), "emoji": "🏖️"},
}

# USD por millón de tokens (entrada, salida). El uso real es gratuito (free tier);
# estos precios de lista permiten simular cuánto costaría la solución en producción.
PRECIOS_USD_POR_MTOK = {
    "openai/gpt-oss-120b": (0.15, 0.60),
    "openai/gpt-oss-20b": (0.075, 0.30),
    "qwen/qwen3.8-27b": (0.29, 0.59),  # referencia: verificar en groq.com/pricing
    "claude-sonnet-5": (2.00, 10.00),
    "claude-haiku-4-5": (1.00, 5.00),
}


def asegurar_api_key() -> None:
    """Pide la API key de Groq si no está en el entorno (nunca se escribe en el código)."""
    if not os.getenv("GROQ_API_KEY"):
        os.environ["GROQ_API_KEY"] = getpass("GROQ_API_KEY: ")

In [ ]:
%%writefile core/schemas.py
"""Modelos Pydantic usados con with_structured_output."""
from typing import Literal, Optional

from pydantic import BaseModel, Field

Destino = Literal["barcelona", "madrid", "mallorca"]
PerfilViajero = Literal[
    "parejas", "familias", "grupos de amigos", "viajeros solos", "trabajo remoto", "viajeros con auto"
]


class AnalisisResenas(BaseModel):
    """Análisis de las reseñas recientes de un alojamiento."""

    sentimiento_general: Literal["muy positivo", "positivo", "mixto", "negativo"]
    puntaje_limpieza: Optional[float] = Field(None, ge=1, le=10, description="1-10; null si las reseñas no lo mencionan")
    puntaje_ubicacion: Optional[float] = Field(None, ge=1, le=10, description="1-10; null si no se menciona")
    puntaje_tranquilidad: Optional[float] = Field(
        None, ge=1, le=10, description="10 = muy silencioso, 1 = muy ruidoso; null si no se menciona"
    )
    puntaje_anfitrion: Optional[float] = Field(None, ge=1, le=10, description="trato y comunicación; null si no se menciona")
    puntaje_fidelidad: Optional[float] = Field(
        None, ge=1, le=10, description="¿el lugar coincide con el anuncio y las fotos?; null si no se menciona"
    )
    pros: list[str] = Field(description="Hasta 3 puntos fuertes, en español, frases cortas")
    contras: list[str] = Field(description="Hasta 3 puntos débiles mencionados; lista vacía si no hay")
    ideal_para: list[PerfilViajero] = Field(description="Perfiles de viajero para los que se recomienda")
    alerta: Optional[str] = Field(None, description="Problema serio y recurrente (seguridad, plagas, estafa); null si no hay")
    resumen: str = Field(description="Una o dos frases en español que resuman la experiencia de los huéspedes")
    cita_textual: str = Field(description="Fragmento literal (sin traducir) de una reseña que respalde el resumen")


class TramoViaje(BaseModel):
    destino: Destino
    check_in: str = Field(description="Fecha de entrada YYYY-MM-DD")
    check_out: str = Field(description="Fecha de salida YYYY-MM-DD")


class SolicitudViaje(BaseModel):
    """Pedido de viaje interpretado a partir del lenguaje natural del usuario."""

    tramos: list[TramoViaje]
    huespedes: int = Field(ge=1, le=16)
    presupuesto_max_noche: Optional[float] = Field(None, description="Euros por noche; null si no se indicó")
    preferencias: list[str] = Field(default_factory=list, description="p. ej. 'tranquilo', 'cerca de la playa'")


class PalabrasClave(BaseModel):
    """Términos para buscar en reseñas escritas en varios idiomas."""

    palabras: list[str] = Field(
        description="8 a 15 palabras o raíces cortas relacionadas con la pregunta, en español, inglés, francés, "
                    "alemán, italiano y catalán (p. ej. 'ruid', 'noise', 'bruit', 'laut')"
    )


class Cita(BaseModel):
    texto: str = Field(description="Fragmento literal de la reseña, en su idioma original, máximo 30 palabras")
    fecha: str = Field(description="Fecha de la reseña YYYY-MM-DD")


class RespuestaResenas(BaseModel):
    """Respuesta a una pregunta sobre un alojamiento, fundamentada en reseñas reales."""

    veredicto: Literal["sí", "no", "mixto", "sin información"]
    respuesta: str = Field(description="2 a 4 frases en español que respondan la pregunta usando solo las reseñas")
    evidencia: list[Cita] = Field(description="1 a 3 citas textuales que respaldan la respuesta; vacía si no hay")

In [ ]:
%%writefile core/metrics.py
"""Registro de tokens, tiempos y costo simulado de cada llamada al LLM."""
import time
from dataclasses import dataclass, field
from typing import Any

from langchain_core.callbacks import BaseCallbackHandler
from langchain_core.outputs import LLMResult

from core.config import PRECIOS_USD_POR_MTOK


def costo_usd(modelo: str, tokens_in: int, tokens_out: int) -> float:
    precio_in, precio_out = PRECIOS_USD_POR_MTOK.get(modelo, (0.0, 0.0))
    return (tokens_in * precio_in + tokens_out * precio_out) / 1_000_000


@dataclass
class RegistroLLM:
    modelo: str
    tokens_in: int
    tokens_out: int
    segundos: float

    @property
    def costo(self) -> float:
        return costo_usd(self.modelo, self.tokens_in, self.tokens_out)


@dataclass(eq=False)  # eq=False: LangChain necesita que el callback sea hasheable
class MetricsTracker(BaseCallbackHandler):
    """Callback que acumula el uso de cada llamada al modelo (sirve para batch, cadenas y agente)."""

    registros: list[RegistroLLM] = field(default_factory=list)
    _inicios: dict = field(default_factory=dict)

    def on_chat_model_start(self, serialized, messages, *, run_id, **kwargs: Any) -> None:
        self._inicios[run_id] = time.perf_counter()

    def on_llm_start(self, serialized, prompts, *, run_id, **kwargs: Any) -> None:
        self._inicios[run_id] = time.perf_counter()

    def on_llm_end(self, response: LLMResult, *, run_id, **kwargs: Any) -> None:
        segundos = time.perf_counter() - self._inicios.pop(run_id, time.perf_counter())
        for generaciones in response.generations:
            for gen in generaciones:
                msg = getattr(gen, "message", None)
                uso = getattr(msg, "usage_metadata", None) or {}
                modelo = (getattr(msg, "response_metadata", {}) or {}).get("model_name", "desconocido")
                self.registros.append(
                    RegistroLLM(modelo, uso.get("input_tokens", 0), uso.get("output_tokens", 0), segundos)
                )

    def resumen(self, n_unidades: int | None = None) -> dict:
        """n_unidades: registros procesados o consultas, para calcular el costo promedio."""
        tin = sum(r.tokens_in for r in self.registros)
        tout = sum(r.tokens_out for r in self.registros)
        costo = sum(r.costo for r in self.registros)
        n = n_unidades or max(len(self.registros), 1)
        return {
            "llamadas": len(self.registros),
            "tokens_entrada": tin,
            "tokens_salida": tout,
            "tokens_totales": tin + tout,
            "segundos_llm": round(sum(r.segundos for r in self.registros), 2),
            "costo_total_usd": round(costo, 6),
            "costo_promedio_usd": round(costo / n, 6),
            "costo_si_fuera_claude_sonnet_usd": round(costo_usd("claude-sonnet-5", tin, tout), 4),
        }

    def reset(self) -> None:
        self.registros.clear()

In [ ]:
%%writefile data/download_data.py
"""Descarga los datos de Inside Airbnb (https://insideairbnb.com/get-the-data/) para los tres destinos."""
from pathlib import Path
import urllib.request

BASE = "https://data.insideairbnb.com/spain"
CIUDADES = {
    "barcelona": "catalonia/barcelona/2026-06-24",
    "madrid": "comunidad-de-madrid/madrid/2026-06-20",
    "mallorca": "islas-baleares/mallorca/2026-06-23",
}
ARCHIVOS = ["listings", "calendar", "reviews"]
RAW = Path(__file__).parent / "raw"


def descargar(forzar: bool = False) -> None:
    RAW.mkdir(parents=True, exist_ok=True)
    for ciudad, ruta in CIUDADES.items():
        for archivo in ARCHIVOS:
            destino = RAW / f"{ciudad}_{archivo}.csv.gz"
            if destino.exists() and not forzar:
                continue
            url = f"{BASE}/{ruta}/data/{archivo}.csv.gz"
            print(f"Descargando {url}")
            urllib.request.urlretrieve(url, destino)
    print("Listo:", sorted(p.name for p in RAW.glob("*.csv.gz")))


if __name__ == "__main__":
    descargar()

In [ ]:
%%writefile core/data_prep.py
"""Construye el catálogo curado de alojamientos (pool) a partir de los CSV crudos de Inside Airbnb.

Salidas en data/processed/:
- pool.parquet          alojamientos activos seleccionados por destino
- calendar_pool.parquet disponibilidad diaria de los alojamientos del pool
- reviews_pool.parquet  últimas reseñas de cada alojamiento del pool (entrada del batch NLP)
- reviews_qa.parquet    hasta 40 reseñas por alojamiento (base de la tool "preguntale a las reseñas")
- price_ref.parquet     precio mediano por destino, barrio y capacidad (referencia de relación precio/calidad)
"""
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = Path(__file__).resolve().parents[1]
RAW = ROOT / "data" / "raw"
PROCESSED = ROOT / "data" / "processed"

CIUDADES = ["barcelona", "madrid", "mallorca"]
POOL_POR_CIUDAD = 500
RESENAS_POR_ALOJAMIENTO = 8
RESENAS_QA = 40
MAX_CHARS_QA = 600
MAX_CHARS_RESENA = 300
FECHA_ACTIVIDAD = "2026-01-01"
SEED = 42

COLUMNAS = [
    "id", "name", "listing_url", "picture_url", "neighbourhood_cleansed", "neighbourhood_group_cleansed",
    "latitude", "longitude", "property_type", "room_type", "accommodates", "bedrooms", "beds",
    "bathrooms_text", "amenities", "price", "minimum_nights", "number_of_reviews", "number_of_reviews_ltm",
    "last_review", "review_scores_rating", "review_scores_accuracy", "review_scores_cleanliness",
    "review_scores_checkin", "review_scores_communication", "review_scores_location", "review_scores_value",
    "host_name", "host_is_superhost", "instant_bookable", "description",
]


def bucket_capacidad(n: pd.Series) -> pd.Series:
    return pd.cut(n, bins=[0, 2, 4, 6, 100], labels=["1-2", "3-4", "5-6", "7+"])


def cargar_listings(ciudad: str) -> pd.DataFrame:
    df = pd.read_csv(RAW / f"{ciudad}_listings.csv.gz", usecols=COLUMNAS)
    df["price"] = pd.to_numeric(df["price"].astype(str).str.replace(r"[$,€]", "", regex=True), errors="coerce")
    df["last_review"] = pd.to_datetime(df["last_review"])
    df["destino"] = ciudad
    df["barrio"] = df["neighbourhood_cleansed"]
    df["cap_bucket"] = bucket_capacidad(df["accommodates"])
    return df


def filtrar_activos(df: pd.DataFrame) -> pd.DataFrame:
    """Alojamientos con actividad reciente, precio publicado y suficientes reseñas para analizarlas."""
    p99 = df["price"].quantile(0.99)
    return df[
        (df["number_of_reviews"] >= 15)
        & (df["last_review"] >= FECHA_ACTIVIDAD)
        & df["price"].between(30, p99)
        & df["review_scores_rating"].notna()
        & (df["room_type"] != "Shared room")
    ]


def muestrear_pool(activos: pd.DataFrame, n: int) -> pd.DataFrame:
    """Muestra estratificada por capacidad y quintil de precio para cubrir todo tipo de viaje."""
    activos = activos.copy()
    activos["price_q"] = pd.qcut(activos["price"], 5, labels=False, duplicates="drop")
    por_celda = int(np.ceil(n / (activos["cap_bucket"].nunique() * activos["price_q"].nunique())))
    muestra = (
        activos.sample(frac=1, random_state=SEED)
        .groupby(["cap_bucket", "price_q"], observed=True).head(por_celda)
    )
    if len(muestra) < n:  # completar celdas vacías (p. ej. pocos monoambientes en Mallorca)
        resto = activos.drop(muestra.index)
        muestra = pd.concat([muestra, resto.sample(min(len(resto), n - len(muestra)), random_state=SEED)])
    return muestra.sample(min(n, len(muestra)), random_state=SEED).drop(columns="price_q")


def construir() -> None:
    PROCESSED.mkdir(parents=True, exist_ok=True)
    pools, refs, calendarios, resenas, resenas_qa = [], [], [], [], []
    for ciudad in CIUDADES:
        listings = cargar_listings(ciudad)
        activos = filtrar_activos(listings)
        refs.append(
            activos.groupby(["destino", "barrio", "cap_bucket"], observed=True)["price"]
            .agg(precio_mediano="median", n="size").reset_index()
        )
        pool = muestrear_pool(activos, POOL_POR_CIUDAD)
        pools.append(pool)
        ids = set(pool["id"])

        cal = pd.read_csv(RAW / f"{ciudad}_calendar.csv.gz", usecols=["listing_id", "date", "available", "minimum_nights"])
        cal = cal[cal["listing_id"].isin(ids)]
        cal["available"] = cal["available"].eq("t")
        calendarios.append(cal)

        rev = pd.read_csv(RAW / f"{ciudad}_reviews.csv.gz", usecols=["listing_id", "id", "date", "comments"])
        rev = rev[rev["listing_id"].isin(ids)].dropna(subset=["comments"])
        rev["comments"] = (
            rev["comments"].str.replace(r"<br\s*/?>", " ", regex=True).str.replace(r"\s+", " ", regex=True).str.strip()
        )
        rev = rev[rev["comments"].str.len() >= 20].sort_values("date").rename(columns={"id": "review_id"})
        qa = rev.groupby("listing_id").tail(RESENAS_QA)
        resenas_qa.append(qa.assign(comments=qa["comments"].str[:MAX_CHARS_QA]))
        ultimas = rev.groupby("listing_id").tail(RESENAS_POR_ALOJAMIENTO)
        resenas.append(ultimas.assign(comments=ultimas["comments"].str[:MAX_CHARS_RESENA]))
        print(f"{ciudad}: {len(listings)} listings → {len(activos)} activos → pool {len(pool)}")

    pool = pd.concat(pools, ignore_index=True)
    pool["cap_bucket"] = pool["cap_bucket"].astype(str)
    pool.to_parquet(PROCESSED / "pool.parquet", index=False)
    ref = pd.concat(refs, ignore_index=True)
    ref["cap_bucket"] = ref["cap_bucket"].astype(str)
    ref.to_parquet(PROCESSED / "price_ref.parquet", index=False)
    pd.concat(calendarios, ignore_index=True).to_parquet(PROCESSED / "calendar_pool.parquet", index=False)
    pd.concat(resenas, ignore_index=True).to_parquet(PROCESSED / "reviews_pool.parquet", index=False)
    pd.concat(resenas_qa, ignore_index=True).to_parquet(PROCESSED / "reviews_qa.parquet", index=False)
    print("Archivos generados en", PROCESSED)


if __name__ == "__main__":
    construir()

In [ ]:
%%writefile core/nlp.py
"""Cadenas LCEL: análisis multidimensional de reseñas (RunnableParallel + batch) e interpretación de pedidos."""
import re
from collections import Counter
from datetime import date

import pandas as pd
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.rate_limiters import InMemoryRateLimiter
from langchain_core.runnables import RunnableLambda, RunnableParallel, RunnablePassthrough
from langchain_groq import ChatGroq

from core.config import MODELO_AGENTE, MODELO_BATCH, PROCESSED
from core.schemas import AnalisisResenas, PalabrasClave, RespuestaResenas, SolicitudViaje

ENRIQUECIDO = PROCESSED / "reviews_enriched.parquet"

# El free tier de Groq permite 8K tokens/minuto por modelo: ~1K tokens por alojamiento → ~6 llamadas/min.
limitador_batch = InMemoryRateLimiter(requests_per_second=0.1, check_every_n_seconds=0.5, max_bucket_size=1)


def llm_batch() -> ChatGroq:
    return ChatGroq(model=MODELO_BATCH, temperature=0, reasoning_effort="low", max_retries=6,
                    rate_limiter=limitador_batch)


def llm_agente(**kwargs) -> ChatGroq:
    kwargs = {"max_retries": 4} | kwargs
    return ChatGroq(model=MODELO_AGENTE, temperature=0.2, reasoning_effort="low", **kwargs)


MODELO_RESPALDO = "qwen/qwen3.8-27b"


def estructurado(schema, llm=None, rate_limiter=None):
    """LLM con salida estructurada. Sin llm explícito arma una cascada de modelos gratuitos (cada uno con su propia
    cuota en Groq): si el primero agota su cuota por minuto o por día, la cadena sigue con el siguiente."""
    if llm is not None:
        return llm.with_structured_output(schema)
    modelos = [
        ChatGroq(model=MODELO_BATCH, temperature=0, reasoning_effort="low", max_retries=0, rate_limiter=rate_limiter),
        ChatGroq(model=MODELO_RESPALDO, temperature=0, max_tokens=900, max_retries=0),
        ChatGroq(model=MODELO_AGENTE, temperature=0, reasoning_effort="low", max_retries=2),
    ]
    principal, *respaldos = [m.with_structured_output(schema) for m in modelos]
    return principal.with_fallbacks(respaldos)


# ---------------------------------------------------------------- análisis de reseñas

PROMPT_RESENAS = ChatPromptTemplate.from_messages([
    ("system",
     "Sos un analista de hospitalidad. Recibís reseñas reales de huéspedes de un alojamiento de Airbnb "
     "en {destino}, escritas en distintos idiomas.\n"
     "Reglas:\n"
     "- Basate EXCLUSIVAMENTE en el texto de las reseñas; no inventes datos ni supongas.\n"
     "- Si un aspecto no se menciona, dejá su puntaje en null.\n"
     "- Los puntajes van de 1 a 10 con un decimal.\n"
     "- Pros, contras, alerta y resumen SIEMPRE en español neutro: traducí lo que esté en otros idiomas.\n"
     "- La cita_textual debe copiarse literal de alguna reseña, en su idioma original, máximo 25 palabras."),
    ("human", "Alojamiento: {nombre}\n\nReseñas:\n{resenas}"),
])

LEXICO = {
    "ruido": r"\b(ruid|noise|noisy|loud|laut|lärm|bruit|bruyant|rumor)",
    "transporte": r"\b(metro|subte|subway|u-bahn|bus|estaci|station|tren|train|tram)",
    "playa": r"\b(playa|platja|beach|strand|plage|spiaggia|mar |sea)",
    "limpieza": r"\b(limpi|clean|sauber|propre|pulit)",
    "suciedad": r"\b(suci|dirty|schmutz|sale |sporc)",
    "wifi": r"\b(wifi|wi-fi|internet)",
}
IDIOMAS = {
    "es": {"el", "la", "muy", "con", "que", "todo", "para", "piso", "estancia"},
    "en": {"the", "and", "was", "very", "great", "place", "stay", "with"},
    "de": {"und", "die", "der", "sehr", "war", "wir", "wohnung", "ist"},
    "fr": {"et", "très", "le", "les", "nous", "appartement", "était", "séjour"},
    "it": {"molto", "e", "il", "della", "casa", "appartamento", "ottimo", "tutto"},
    "ca": {"molt", "i", "amb", "pis", "tot", "estada", "gràcies"},
    "nl": {"en", "het", "een", "zeer", "was", "mooi", "fijn", "prima"},
    "sv": {"och", "mycket", "var", "vi", "boende", "fint", "bra"},
}


def detectar_idioma(texto: str) -> str:
    palabras = set(re.findall(r"\w+", texto.lower()))
    puntajes = {idioma: len(palabras & vocab) for idioma, vocab in IDIOMAS.items()}
    mejor = max(puntajes, key=puntajes.get)
    return mejor if puntajes[mejor] >= 2 else "otro"


def senales_lexicas(entrada: dict) -> dict:
    """Análisis sin LLM: menciones de temas clave por léxico multilingüe."""
    texto = entrada["resenas"].lower()
    return {tema: len(re.findall(patron, texto)) for tema, patron in LEXICO.items()}


def estadisticas(entrada: dict) -> dict:
    """Análisis sin LLM: idiomas y volumen del texto analizado."""
    lineas = [l for l in entrada["resenas"].split("\n") if l.strip()]
    idiomas = Counter(detectar_idioma(l) for l in lineas)
    return {"n_resenas": len(lineas), "idiomas": dict(idiomas), "caracteres": len(entrada["resenas"])}


def cadena_analisis(llm=None, limitar: bool = True):
    """RunnableParallel: tres análisis sobre la misma entrada (LLM estructurado + léxico + estadísticas)."""
    return RunnableParallel(
        nlp=PROMPT_RESENAS | estructurado(AnalisisResenas, llm, limitador_batch if limitar else None),
        senales=RunnableLambda(senales_lexicas),
        stats=RunnableLambda(estadisticas),
    )


def preparar_entradas(pool: pd.DataFrame, resenas: pd.DataFrame) -> list[dict]:
    agrupadas = resenas.groupby("listing_id")["comments"].apply(lambda s: "\n".join(f"- {c}" for c in s))
    return [
        {"listing_id": int(r.id), "destino": r.destino.title(), "nombre": r.name, "resenas": agrupadas.get(r.id, "")}
        for r in pool.itertuples()
        if r.id in agrupadas.index
    ]


def aplanar(entrada: dict, salida: dict | Exception) -> dict:
    fila = {"listing_id": entrada["listing_id"]}
    if isinstance(salida, Exception):
        return fila | {"error": str(salida)[:200]}
    nlp: AnalisisResenas = salida["nlp"]
    fila |= nlp.model_dump()
    fila |= {f"menciones_{k}": v for k, v in salida["senales"].items()}
    fila |= {"n_resenas": salida["stats"]["n_resenas"], "idiomas": salida["stats"]["idiomas"]}
    return fila


def enriquecer(entradas: list[dict], callbacks=None, max_concurrency: int = 2, llm=None,
               limitar: bool = True) -> pd.DataFrame:
    """Procesa con .batch() y devuelve un DataFrame enriquecido. Los errores quedan registrados, no cortan el batch."""
    cadena = cadena_analisis(llm, limitar)
    salidas = cadena.batch(
        entradas, config={"max_concurrency": max_concurrency, "callbacks": callbacks or []}, return_exceptions=True
    )
    return pd.DataFrame([aplanar(e, s) for e, s in zip(entradas, salidas)])


def cargar_enriquecido() -> pd.DataFrame:
    if ENRIQUECIDO.exists():
        return pd.read_parquet(ENRIQUECIDO)
    return pd.DataFrame(columns=["listing_id"])


def guardar_enriquecido(nuevo: pd.DataFrame) -> pd.DataFrame:
    """Acumula resultados (cache): permite correr el batch en tandas respetando la cuota diaria."""
    previo = cargar_enriquecido()
    ok = nuevo[nuevo.get("error").isna()] if "error" in nuevo else nuevo
    total = pd.concat([previo, ok], ignore_index=True).drop_duplicates("listing_id", keep="last")
    total["idiomas"] = total["idiomas"].astype(str)
    total.to_parquet(ENRIQUECIDO, index=False)
    return total


# ---------------------------------------------------------------- interpretación del pedido

PROMPT_SOLICITUD = ChatPromptTemplate.from_messages([
    ("system",
     "Convertís pedidos de viaje en datos estructurados. Hoy es {hoy}. Destinos válidos: barcelona, madrid, mallorca.\n"
     "- Si el usuario no indica el año, usá la próxima ocurrencia futura de esa fecha.\n"
     "- Si indica cantidad de noches en vez de fecha de salida, calculala.\n"
     "- Si los tramos son consecutivos, el check_out de uno es el check_in del siguiente.\n"
     "- No inventes presupuesto ni preferencias que el usuario no mencionó."),
    ("human", "{pedido}"),
])


def cadena_solicitud(llm=None):
    llm = llm or llm_agente()
    return (
        RunnableLambda(lambda pedido: {"pedido": pedido, "hoy": date.today().isoformat()})
        | PROMPT_SOLICITUD
        | llm.with_structured_output(SolicitudViaje)
    )


# ---------------------------------------------------------------- preguntas sobre las reseñas (RAG liviano)

PROMPT_PALABRAS = ChatPromptTemplate.from_messages([
    ("system", "Generás términos de búsqueda para encontrar reseñas de alojamientos que hablen de un tema. "
               "Usá raíces cortas que funcionen como subcadenas e incluí al menos 2 términos en cada idioma: "
               "español, inglés, francés, alemán e italiano (p. ej. 'limpi', 'clean', 'propre', 'sauber', 'pulit')."),
    ("human", "Pregunta del viajero: {pregunta}"),
])

PROMPT_RESPUESTA = ChatPromptTemplate.from_messages([
    ("system",
     "Respondés preguntas de viajeros sobre un alojamiento usando EXCLUSIVAMENTE las reseñas provistas.\n"
     "- El veredicto responde LITERALMENTE la pregunta: ante '¿tiene problemas de limpieza?' sin quejas → 'no'; "
     "ante '¿es tranquilo?' con elogios al silencio → 'sí'. Si hay opiniones divididas → 'mixto'.\n"
     "- Si las reseñas no mencionan el tema, el veredicto es 'sin información' y lo decís claramente.\n"
     "- No generalices a partir de una sola reseña: indicá cuántas lo mencionan.\n"
     "- Las citas deben copiarse literalmente de las reseñas, con su fecha."),
    ("human", "Alojamiento: {nombre}\nPregunta: {pregunta}\n\n"
              "Reseñas relevantes ({n_relevantes} de {n_total} analizadas):\n{fragmentos}"),
])


def _recuperar(entrada: dict, k: int = 12) -> dict:
    """Recupera las reseñas con más coincidencias de palabras clave (si no hay, las más recientes)."""
    resenas: pd.DataFrame = entrada["resenas"]
    palabras = [p.lower().strip() for p in entrada["palabras"].palabras if len(p.strip()) >= 3]
    texto = resenas["comments"].str.lower()
    coincidencias = sum(texto.str.count(re.escape(p)) for p in palabras) if palabras else 0
    relevantes = resenas.assign(score=coincidencias).query("score > 0")
    elegidas = relevantes.sort_values(["score", "date"], ascending=False).head(k)
    if elegidas.empty:
        elegidas = resenas.sort_values("date", ascending=False).head(k)
    fragmentos = "\n".join(f"- [{r.date}] {r.comments}" for r in elegidas.itertuples())
    return {"fragmentos": fragmentos, "n_relevantes": len(relevantes), "n_total": len(resenas)}


def cadena_preguntas(llm=None):
    """LCEL: expandir la pregunta a palabras clave multilingües → recuperar reseñas → responder con evidencia."""
    expandir = PROMPT_PALABRAS | estructurado(PalabrasClave, llm)
    return (
        RunnablePassthrough.assign(palabras=expandir)
        | RunnablePassthrough.assign(recuperado=RunnableLambda(_recuperar))
        | RunnableLambda(lambda x: {**x, **x["recuperado"]})
        | RunnableParallel(
            respuesta=PROMPT_RESPUESTA | estructurado(RespuestaResenas, llm),
            n_relevantes=RunnableLambda(lambda x: x["n_relevantes"]),
            n_total=RunnableLambda(lambda x: x["n_total"]),
            palabras=RunnableLambda(lambda x: x["palabras"].palabras),
        )
    )

In [ ]:
%%writefile core/scoring.py
"""Puntaje propio (1-10) y selección de las 4 opciones por destino."""
from datetime import date
from functools import lru_cache

import numpy as np
import pandas as pd

from core.config import DESTINOS, PROCESSED
from core.data_prep import bucket_capacidad
from core.nlp import cargar_enriquecido

PESOS = {"calidad": 0.30, "resenas_nlp": 0.30, "precio_calidad": 0.20, "ubicacion": 0.10, "ajuste_capacidad": 0.10}
M_BAYES = 20  # reseñas "virtuales" con el promedio del destino: evita que 5.0 con pocas reseñas gane


@lru_cache(maxsize=1)
def datos() -> dict[str, pd.DataFrame]:
    pool = pd.read_parquet(PROCESSED / "pool.parquet")
    cal = pd.read_parquet(PROCESSED / "calendar_pool.parquet")
    cal["date"] = pd.to_datetime(cal["date"])
    ref = pd.read_parquet(PROCESSED / "price_ref.parquet")
    nlp = cargar_enriquecido()
    pool = pool.merge(nlp, left_on="id", right_on="listing_id", how="left")
    return {"pool": _puntajes_base(pool, ref), "cal": cal}


def recargar() -> None:
    datos.cache_clear()


def _clip01(x):
    return np.clip(x, 0, 1)


def _puntajes_base(pool: pd.DataFrame, ref: pd.DataFrame) -> pd.DataFrame:
    """Componentes que no dependen del pedido del usuario."""
    df = pool.copy()
    media_destino = df.groupby("destino")["review_scores_rating"].transform("mean")
    v = df["number_of_reviews"]
    df["rating_bayes"] = (v * df["review_scores_rating"] + M_BAYES * media_destino) / (v + M_BAYES)
    df["c_calidad"] = _clip01((df["rating_bayes"] - 4.0) / 1.0)

    aspectos = ["puntaje_limpieza", "puntaje_ubicacion", "puntaje_tranquilidad", "puntaje_anfitrion", "puntaje_fidelidad"]
    for a in aspectos:
        if a not in df:
            df[a] = np.nan
    sentimiento = df.get("sentimiento_general", pd.Series(index=df.index, dtype=object)).map(
        {"muy positivo": 1.0, "positivo": 0.8, "mixto": 0.45, "negativo": 0.1}
    )
    promedio_aspectos = df[aspectos].mean(axis=1) / 10
    # Sin análisis NLP disponible se usan las subcategorías numéricas de Airbnb como respaldo.
    respaldo = df[["review_scores_cleanliness", "review_scores_communication", "review_scores_accuracy"]].mean(axis=1) / 5
    df["c_resenas_nlp"] = (0.6 * promedio_aspectos + 0.4 * sentimiento).fillna(respaldo)
    df["c_resenas_nlp"] = df["c_resenas_nlp"] - np.where(df.get("alerta").notna() if "alerta" in df else False, 0.25, 0)
    df["c_resenas_nlp"] = _clip01(df["c_resenas_nlp"])
    df["tiene_nlp"] = df["sentimiento_general"].notna() if "sentimiento_general" in df else False

    df = df.merge(ref[["destino", "barrio", "cap_bucket", "precio_mediano"]], on=["destino", "barrio", "cap_bucket"], how="left")
    por_capacidad = ref.groupby(["destino", "cap_bucket"])["precio_mediano"].median().rename("mediana_cap")
    df = df.join(por_capacidad, on=["destino", "cap_bucket"])
    df["precio_ref"] = df["precio_mediano"].fillna(df["mediana_cap"])
    ratio = df["price"] / df["precio_ref"]
    # Relación precio/calidad: barato respecto de su zona y con buen rating puntúa alto.
    df["c_precio_calidad"] = _clip01(1 - (ratio - 0.5) / 1.5) * (0.5 + 0.5 * df["c_calidad"])

    ubic_nlp = df["puntaje_ubicacion"] / 10
    ubic_airbnb = _clip01((df["review_scores_location"] - 4.0) / 1.0)
    df["c_ubicacion"] = ubic_airbnb.where(ubic_nlp.isna(), 0.5 * ubic_airbnb + 0.5 * ubic_nlp).fillna(0.5)
    return df


def _ajuste_capacidad(capacidad: pd.Series, huespedes: int) -> pd.Series:
    exceso = capacidad - huespedes
    return _clip01(1 - 0.2 * (exceso - 1).clip(lower=0))


def normalizar_pesos(pesos: dict | None) -> dict:
    """Acepta pesos en cualquier escala (p. ej. sliders 0-10) y los lleva a suma 1."""
    pesos = {k: float((pesos or {}).get(k, v)) for k, v in PESOS.items()}
    total = sum(pesos.values())
    return {k: v / total for k, v in pesos.items()} if total > 0 else dict(PESOS)


def puntuar(df: pd.DataFrame, huespedes: int, pesos: dict | None = None) -> pd.DataFrame:
    df = df.copy()
    pesos = normalizar_pesos(pesos)
    df["c_ajuste_capacidad"] = _ajuste_capacidad(df["accommodates"], huespedes)
    total = sum(pesos[k] * df[f"c_{k}"] for k in PESOS)
    df["puntaje"] = (1 + 9 * total).round(1)
    df["rating_airbnb_10"] = (df["review_scores_rating"] * 2).round(1)
    return df


def disponibles(destino: str, check_in: str, check_out: str) -> set[int]:
    """IDs disponibles todas las noches del rango y cuya estadía mínima permite la reserva."""
    cal = datos()["cal"]
    ini, fin = pd.Timestamp(check_in), pd.Timestamp(check_out)
    noches = (fin - ini).days
    ids_destino = set(datos()["pool"].query("destino == @destino")["id"])
    rango = cal[cal["listing_id"].isin(ids_destino) & (cal["date"] >= ini) & (cal["date"] < fin)]
    completos = rango.groupby("listing_id").agg(n=("available", "size"), libres=("available", "sum"))
    ok = completos[(completos["n"] == noches) & (completos["libres"] == noches)].index
    minimo = cal[(cal["date"] == ini) & cal["listing_id"].isin(ok)].set_index("listing_id")["minimum_nights"]
    return set(minimo[minimo <= noches].index)


def candidatos(destino: str, check_in: str, check_out: str, huespedes: int,
               presupuesto_max_noche: float | None = None, pesos: dict | None = None) -> pd.DataFrame:
    destino = destino.lower().strip()
    if destino not in DESTINOS:
        raise ValueError(f"Destino '{destino}' no disponible. Opciones: {', '.join(DESTINOS)}")
    if date.fromisoformat(check_out) <= date.fromisoformat(check_in):
        raise ValueError("La fecha de salida debe ser posterior a la de entrada.")
    pool = datos()["pool"]
    df = pool[(pool["destino"] == destino) & (pool["accommodates"] >= huespedes)]
    df = df[df["id"].isin(disponibles(destino, check_in, check_out))]
    if presupuesto_max_noche:
        df = df[df["price"] <= presupuesto_max_noche]
    return puntuar(df, huespedes, pesos)


def elegir_cuatro(df: pd.DataFrame) -> list[tuple[str, pd.Series]]:
    """Recomendada por el modelo, mejor rateada, más barata y más cara (siempre alojamientos distintos)."""
    elegidos, usados = [], set()
    # Evita proponer, por ejemplo, una casa para 15 personas a un grupo de 3 (hasta 3 plazas de más).
    razonables = df[df["c_ajuste_capacidad"] >= 0.6]
    if len(razonables) >= 4:
        df = razonables

    def tomar(etiqueta, ordenado):
        for _, fila in ordenado.iterrows():
            if fila["id"] not in usados:
                usados.add(fila["id"])
                elegidos.append((etiqueta, fila))
                return

    tomar("recomendada", df.sort_values(["puntaje", "number_of_reviews"], ascending=False))
    tomar("mejor_rateada", df.sort_values(["rating_bayes", "number_of_reviews"], ascending=False))
    tomar("mas_barata", df.sort_values("price"))
    tomar("mas_cara", df.sort_values("price", ascending=False))
    return elegidos


def detalle_puntaje(fila: pd.Series) -> dict:
    return {k: round(float(fila[f"c_{k}"]) * 10, 1) for k in PESOS}

In [ ]:
%%writefile core/lugares.py
"""Puntos de interés por destino y distancias desde cada alojamiento (fórmula de haversine)."""
import unicodedata

import numpy as np
import pandas as pd

LUGARES = {
    "barcelona": {
        "Sagrada Família": (41.4036, 2.1744),
        "Park Güell": (41.4145, 2.1527),
        "Passeig de Gràcia / Casa Batlló": (41.3917, 2.1650),
        "Barrio Gótico / Catedral": (41.3840, 2.1762),
        "Plaça de Catalunya": (41.3870, 2.1701),
        "Playa de la Barceloneta": (41.3784, 2.1925),
        "Camp Nou": (41.3809, 2.1228),
        "Aeropuerto El Prat": (41.2974, 2.0833),
    },
    "madrid": {
        "Puerta del Sol": (40.4169, -3.7035),
        "Plaza Mayor": (40.4155, -3.7074),
        "Museo del Prado": (40.4138, -3.6921),
        "Palacio Real": (40.4180, -3.7143),
        "Parque del Retiro": (40.4153, -3.6845),
        "Gran Vía": (40.4200, -3.7058),
        "Estadio Santiago Bernabéu": (40.4531, -3.6883),
        "Aeropuerto Barajas": (40.4983, -3.5676),
    },
    "mallorca": {
        "Catedral de Palma (casco antiguo)": (39.5676, 2.6484),
        "Sóller": (39.7667, 2.7150),
        "Valldemossa": (39.7106, 2.6225),
        "Pollença": (39.8770, 3.0160),
        "Playa de Alcúdia": (39.8390, 3.1270),
        "Cala Millor": (39.5960, 3.3830),
        "Playa Es Trenc": (39.3450, 2.9860),
        "Cuevas del Drach": (39.5360, 3.3300),
        "Aeropuerto de Palma": (39.5517, 2.7388),
    },
}
AEROPUERTOS = {"barcelona": "Aeropuerto El Prat", "madrid": "Aeropuerto Barajas", "mallorca": "Aeropuerto de Palma"}
RADIO_CERCA_KM = {"barcelona": 1.5, "madrid": 1.5, "mallorca": 12.0}


def _normalizar(texto: str) -> str:
    texto = unicodedata.normalize("NFKD", texto.lower())
    return "".join(c for c in texto if not unicodedata.combining(c))


def haversine_km(lat1, lon1, lat2, lon2):
    lat1, lon1, lat2, lon2 = map(np.radians, (lat1, lon1, lat2, lon2))
    a = np.sin((lat2 - lat1) / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin((lon2 - lon1) / 2) ** 2
    return 6371 * 2 * np.arcsin(np.sqrt(a))


def buscar_lugar(destino: str, texto: str) -> str | None:
    """Encuentra el punto de interés que mejor coincide con lo que escribió el usuario."""
    consulta = _normalizar(texto)
    for nombre in LUGARES.get(destino, {}):
        partes = [_normalizar(p) for p in nombre.replace("(", "/").replace(")", "").split("/")]
        if any(p.strip() and (p.strip() in consulta or consulta in p.strip()) for p in partes):
            return nombre
    return None


def texto_distancia(km: float, destino: str) -> str:
    if destino != "mallorca" and km <= 2.5:
        return f"{max(1, round(km * 12))} min a pie"  # ~5 km/h
    if destino == "mallorca":
        return f"{km:.0f} km (~{max(5, round(km / 50 * 60))} min en auto)"
    return f"{km:.1f} km"


def distancias(df: pd.DataFrame, destino: str) -> pd.DataFrame:
    """Una columna dist_<lugar> en km por cada punto de interés del destino."""
    df = df.copy()
    for nombre, (lat, lon) in LUGARES[destino].items():
        df[f"dist::{nombre}"] = haversine_km(df["latitude"], df["longitude"], lat, lon)
    return df


def cercanias(fila, destino: str, n: int = 2) -> list[dict]:
    """Los n puntos de interés más cercanos (sin contar el aeropuerto) + la distancia al aeropuerto."""
    lat, lon = float(fila["latitude"]), float(fila["longitude"])
    aeropuerto = AEROPUERTOS[destino]
    dist = {nombre: float(haversine_km(lat, lon, *coords)) for nombre, coords in LUGARES[destino].items()}
    cercanos = sorted((k for k in dist if k != aeropuerto), key=dist.get)[:n] + [aeropuerto]
    return [{"lugar": k, "km": round(dist[k], 2), "texto": texto_distancia(dist[k], destino)} for k in cercanos]

In [ ]:
%%writefile core/clima.py
"""Clima esperado para las fechas del viaje con Open-Meteo (gratuito, sin API key).

- Si el viaje empieza dentro de los próximos 14 días: pronóstico real.
- Si es más adelante: referencia histórica de las mismas fechas del año anterior.
"""
from datetime import date, timedelta

import requests

from core.config import DESTINOS

FORECAST = "https://api.open-meteo.com/v1/forecast"
ARCHIVO = "https://archive-api.open-meteo.com/v1/archive"
VARIABLES = "temperature_2m_max,temperature_2m_min,precipitation_sum"


def _menos_un_anio(d: date) -> date:
    try:
        return d.replace(year=d.year - 1)
    except ValueError:  # 29 de febrero
        return d - timedelta(days=365)


def clima(destino: str, check_in: str, check_out: str) -> dict:
    lat, lon = DESTINOS[destino]["centro"]
    ini, fin = date.fromisoformat(check_in), date.fromisoformat(check_out)
    hoy = date.today()
    if ini <= hoy + timedelta(days=14) and fin <= hoy + timedelta(days=15):
        url, fuente = FORECAST, "pronóstico"
        params = {"start_date": ini.isoformat(), "end_date": fin.isoformat()}
    else:
        url, fuente = ARCHIVO, f"histórico (mismas fechas de {ini.year - 1})"
        params = {"start_date": _menos_un_anio(ini).isoformat(), "end_date": _menos_un_anio(fin).isoformat()}
    params |= {"latitude": lat, "longitude": lon, "daily": VARIABLES, "timezone": "Europe/Madrid"}
    r = requests.get(url, params=params, timeout=15)
    r.raise_for_status()
    d = r.json()["daily"]
    maximas = [x for x in d["temperature_2m_max"] if x is not None]
    minimas = [x for x in d["temperature_2m_min"] if x is not None]
    lluvia = [x or 0 for x in d["precipitation_sum"]]
    return {
        "destino": destino, "fuente": fuente, "check_in": check_in, "check_out": check_out,
        "max_promedio": round(sum(maximas) / len(maximas), 1), "min_promedio": round(sum(minimas) / len(minimas), 1),
        "lluvia_total_mm": round(sum(lluvia), 1), "dias_con_lluvia": sum(1 for x in lluvia if x >= 1),
        "dias": [{"fecha": f, "max": mx, "min": mn, "lluvia": ll}
                 for f, mx, mn, ll in zip(d["time"], d["temperature_2m_max"], d["temperature_2m_min"], lluvia)],
    }

In [ ]:
%%writefile core/tools.py
"""Tools del agente. Devuelven (contenido para el LLM, artifact para la interfaz) con response_format='content_and_artifact':
el LLM recibe un resumen compacto (ahorra tokens) y Chainlit recibe los datos completos para dibujar tarjetas y gráficos."""
import json
from collections import defaultdict
from datetime import date

import pandas as pd
from langchain.tools import ToolRuntime, tool

from core import lugares, scoring
from core.clima import clima
from core.config import DESTINOS, PROCESSED
from core.nlp import cadena_preguntas, enriquecer, guardar_enriquecido, preparar_entradas

ETIQUETAS_CORTAS = {"recomendada": "🧭 Recomendada", "mejor_rateada": "⭐ Mejor rateada",
                    "mas_barata": "💸 Más barata", "mas_cara": "💎 Más cara"}
ETIQUETAS = {
    "recomendada": "🧭 Recomendada por nuestro modelo",
    "mejor_rateada": "⭐ Mejor rateada",
    "mas_barata": "💸 Más barata",
    "mas_cara": "💎 Más cara",
}
FECHA_RELEVAMIENTO = {"barcelona": "24/06/2026", "madrid": "20/06/2026", "mallorca": "23/06/2026"}

# Estado por sesión (thread_id): plan elegido, últimas opciones mostradas y últimos parámetros de búsqueda
PLANES: dict[str, dict[str, dict]] = defaultdict(dict)
ULTIMAS_OPCIONES: dict[str, dict[str, list[dict]]] = defaultdict(dict)
ULTIMAS_BUSQUEDAS: dict[str, dict[str, dict]] = defaultdict(dict)


def _callbacks(runtime: ToolRuntime | None):
    """Propaga el MetricsTracker de la sesión a las llamadas al LLM que se hacen dentro de las tools."""
    try:
        return runtime.config.get("callbacks")
    except Exception:
        return None


def _thread(runtime: ToolRuntime | None) -> str:
    try:
        return runtime.config["configurable"]["thread_id"]
    except Exception:
        return "default"


def _pesos(runtime: ToolRuntime | None) -> dict | None:
    """Pesos del puntaje elegidos por el usuario con los sliders de la interfaz."""
    try:
        return runtime.config["configurable"].get("pesos")
    except Exception:
        return None


def _limpio(valor):
    if isinstance(valor, float) and pd.isna(valor):
        return None
    if hasattr(valor, "tolist"):
        return valor.tolist()
    return valor


def _id(listing_id) -> int | None:
    """Los ids de Airbnb tienen hasta 19 dígitos: se manejan como texto porque como número JSON pierden precisión."""
    texto = str(listing_id).strip().strip('"').split(".")[0]
    if not texto.isdigit():
        return None
    ids = scoring.datos()["pool"]["id"]
    if int(texto) in set(ids):
        return int(texto)
    # Recuperación si el id llegó redondeado (p. ej. ...712600 en vez de ...712747): coincidencia por prefijo
    candidatos = ids[ids.astype(str).str.startswith(texto[:15])] if len(texto) >= 16 else ids.iloc[0:0]
    return int(candidatos.iloc[0]) if len(candidatos) == 1 else None


def _fila(listing_id) -> pd.Series | None:
    lid = _id(listing_id)
    if lid is None:
        return None
    pool = scoring.datos()["pool"]
    return pool[pool["id"] == lid].iloc[0]


def _tarjeta(etiqueta: str, fila, noches: int, check_in: str, check_out: str, cerca_de: str | None) -> dict:
    campos = ["id", "name", "destino", "barrio", "latitude", "longitude", "room_type", "property_type", "accommodates",
              "bedrooms", "beds", "bathrooms_text", "price", "picture_url", "listing_url", "puntaje", "rating_airbnb_10",
              "number_of_reviews", "host_is_superhost", "sentimiento_general", "resumen", "pros", "contras",
              "ideal_para", "alerta", "cita_textual", "tiene_nlp"]
    t = {c: _limpio(fila.get(c)) for c in campos}
    t["id"] = str(int(t["id"]))
    cerca = lugares.cercanias(fila, t["destino"])
    if cerca_de:
        km = float(lugares.haversine_km(float(fila["latitude"]), float(fila["longitude"]),
                                        *lugares.LUGARES[t["destino"]][cerca_de]))
        cerca = [{"lugar": cerca_de, "km": round(km, 2), "texto": lugares.texto_distancia(km, t["destino"])}] + \
                [c for c in cerca if c["lugar"] != cerca_de][:2]
    t |= {"etiqueta": etiqueta, "titulo_etiqueta": ETIQUETAS[etiqueta],
          "etiqueta_corta": ETIQUETAS_CORTAS[etiqueta], "noches": noches, "check_in": check_in,
          "check_out": check_out, "total": round(float(fila["price"]) * noches, 2),
          "detalle_puntaje": scoring.detalle_puntaje(fila), "cercanias": cerca}
    return t


def _enriquecer_faltantes(filas: list, callbacks=None) -> None:
    """Análisis NLP en el momento (con .batch) para las opciones que todavía no estaban analizadas."""
    faltan = [f for f in filas if not bool(f.get("tiene_nlp"))]
    if not faltan:
        return
    resenas = pd.read_parquet(PROCESSED / "reviews_pool.parquet")
    pool = pd.DataFrame(faltan)[["id", "destino", "name"]]
    entradas = preparar_entradas(pool, resenas[resenas["listing_id"].isin(pool["id"])])
    try:
        guardar_enriquecido(enriquecer(entradas, max_concurrency=4, callbacks=callbacks, limitar=False))
    except Exception as e:  # sin análisis NLP las tarjetas se muestran igual, con el puntaje de respaldo
        print(f"[aviso] no se pudo enriquecer en vivo: {e}")
        return
    scoring.recargar()


def _resumen_llm(t: dict) -> dict:
    datos = {k: t[k] for k in ["etiqueta", "id", "name", "barrio", "accommodates", "price", "total", "puntaje",
                               "rating_airbnb_10", "resumen", "alerta"] if k in t}
    datos["cerca_de"] = [f"{c['lugar']}: {c['texto']}" for c in t.get("cercanias", [])[:2]]
    return datos


def ejecutar_busqueda(destino: str, check_in: str, check_out: str, huespedes: int,
                      presupuesto_max_noche: float | None = None, cerca_de: str | None = None,
                      pesos: dict | None = None, thread: str = "default", callbacks=None) -> tuple[str, dict]:
    """Lógica de búsqueda compartida por la tool y por la interfaz (recalcular al mover los sliders)."""
    destino = destino.lower().strip()
    try:
        df = scoring.candidatos(destino, check_in, check_out, huespedes, presupuesto_max_noche, pesos)
    except ValueError as e:
        return f"Error: {e}", {"tipo": "error", "mensaje": str(e)}

    lugar = None
    if cerca_de:
        lugar = lugares.buscar_lugar(destino, cerca_de)
        if lugar is None:
            opciones = ", ".join(lugares.LUGARES[destino])
            return (f"No reconozco '{cerca_de}' en {destino}. Lugares disponibles: {opciones}.",
                    {"tipo": "error", "mensaje": f"Lugar desconocido: {cerca_de}"})
        lat, lon = lugares.LUGARES[destino][lugar]
        df = df.assign(dist_km=lugares.haversine_km(df["latitude"], df["longitude"], lat, lon))
        cerca = df[df["dist_km"] <= lugares.RADIO_CERCA_KM[destino]]
        df = cerca if len(cerca) >= 4 else df.nsmallest(12, "dist_km")

    noches = (date.fromisoformat(check_out) - date.fromisoformat(check_in)).days
    if df.empty:
        msg = (f"No hay alojamientos disponibles en {destino} del {check_in} al {check_out} para {huespedes} "
               f"huéspedes{' con ese presupuesto' if presupuesto_max_noche else ''}. Sugerí cambiar fechas o presupuesto.")
        return msg, {"tipo": "sin_resultados", "mensaje": msg}

    elegidas = scoring.elegir_cuatro(df)
    _enriquecer_faltantes([f for _, f in elegidas], callbacks)
    ids = [f["id"] for _, f in elegidas]
    actualizados = scoring.puntuar(scoring.datos()["pool"].query("id in @ids"), huespedes, pesos).set_index("id")
    tarjetas = [_tarjeta(et, actualizados.loc[f["id"]].to_dict() | {"id": f["id"]}, noches, check_in, check_out, lugar)
                for et, f in elegidas]
    ULTIMAS_OPCIONES[thread][destino] = tarjetas
    ULTIMAS_BUSQUEDAS[thread][destino] = {"destino": destino, "check_in": check_in, "check_out": check_out,
                                          "huespedes": huespedes, "presupuesto_max_noche": presupuesto_max_noche,
                                          "cerca_de": cerca_de}

    contenido = {
        "destino": destino, "check_in": check_in, "check_out": check_out, "noches": noches, "huespedes": huespedes,
        "cerca_de": lugar, "candidatos_evaluados": len(df), "opciones": [_resumen_llm(t) for t in tarjetas],
        "nota": "La interfaz ya muestra las tarjetas con fotos y mapa; no repitas todos los datos, destacá diferencias.",
    }
    artifact = {"tipo": "opciones", "destino": destino, "huespedes": huespedes, "candidatos": len(df),
                "tarjetas": tarjetas, "cerca_de": lugar, "relevamiento": FECHA_RELEVAMIENTO[destino],
                "pesos": scoring.normalizar_pesos(pesos)}
    return json.dumps(contenido, ensure_ascii=False, default=str), artifact


@tool(response_format="content_and_artifact")
def buscar_opciones(destino: str, check_in: str, check_out: str, huespedes: int,
                    presupuesto_max_noche: float | None = None, cerca_de: str | None = None,
                    runtime: ToolRuntime = None):
    """Busca alojamientos disponibles en un destino (barcelona, madrid o mallorca) para las fechas indicadas
    (YYYY-MM-DD) con capacidad para todos los huéspedes. Devuelve 4 opciones: recomendada por el modelo,
    mejor rateada, más barata y más cara, con puntaje 1-10 y distancia a puntos de interés.
    cerca_de: opcional, un punto de interés (p. ej. 'Sagrada Família', 'Museo del Prado', 'Sóller', 'aeropuerto').
    Usala una vez por cada destino del viaje."""
    return ejecutar_busqueda(destino, check_in, check_out, huespedes, presupuesto_max_noche, cerca_de,
                             _pesos(runtime), _thread(runtime), _callbacks(runtime))


@tool(response_format="content_and_artifact")
def detalle_alojamiento(listing_id: str, runtime: ToolRuntime = None):
    """Devuelve el análisis general de reseñas de un alojamiento: pros, contras, perfiles ideales, alertas,
    cita textual de un huésped y puntajes por aspecto. Usala cuando pidan más información de una opción."""
    fila = _fila(listing_id)
    if fila is None:
        return (f"No encontré el alojamiento {listing_id}. Usá exactamente uno de los ids (texto) devueltos por "
                f"buscar_opciones; no reintentes con el mismo id."), {"tipo": "error"}
    if not bool(fila.get("tiene_nlp")):
        _enriquecer_faltantes([fila], _callbacks(runtime))
        fila = _fila(listing_id)
    info_id = str(int(fila["id"]))
    campos = ["name", "destino", "barrio", "sentimiento_general", "resumen", "pros", "contras", "ideal_para",
              "alerta", "cita_textual", "puntaje_limpieza", "puntaje_ubicacion", "puntaje_tranquilidad",
              "puntaje_anfitrion", "puntaje_fidelidad", "number_of_reviews", "listing_url"]
    info = {"id": info_id} | {c: _limpio(fila.get(c)) for c in campos}
    info["cerca_de"] = lugares.cercanias(fila, fila["destino"], n=3)
    return json.dumps(info, ensure_ascii=False, default=str), {"tipo": "detalle", "info": info}


@tool(response_format="content_and_artifact")
def consultar_resenas(listing_id: str, pregunta: str, runtime: ToolRuntime = None):
    """Responde una pregunta puntual sobre un alojamiento (ruido, ascensor, limpieza, seguridad, cercanía al metro,
    estado del baño, etc.) buscando en hasta 40 reseñas reales en varios idiomas y citándolas como evidencia."""
    fila = _fila(listing_id)
    if fila is None:
        return f"No encontré el alojamiento {listing_id} en el catálogo.", {"tipo": "error"}
    resenas = pd.read_parquet(PROCESSED / "reviews_qa.parquet", filters=[("listing_id", "==", int(fila["id"]))])
    if resenas.empty:
        return "Ese alojamiento no tiene reseñas disponibles.", {"tipo": "error"}
    try:
        salida = cadena_preguntas().invoke(
            {"pregunta": pregunta, "nombre": fila["name"], "resenas": resenas},
            config={"callbacks": _callbacks(runtime)},
        )
    except Exception as e:
        return f"No pude analizar las reseñas en este momento ({type(e).__name__}).", {"tipo": "error"}
    r = salida["respuesta"]
    info = {"listing_id": str(int(fila["id"])), "nombre": fila["name"], "pregunta": pregunta, "veredicto": r.veredicto,
            "respuesta": r.respuesta, "evidencia": [c.model_dump() for c in r.evidencia],
            "n_relevantes": salida["n_relevantes"], "n_total": salida["n_total"], "palabras": salida["palabras"]}
    contenido = {k: info[k] for k in ["veredicto", "respuesta", "evidencia", "n_relevantes", "n_total"]}
    return json.dumps(contenido, ensure_ascii=False), {"tipo": "respuesta_resenas", **info}


@tool(response_format="content_and_artifact")
def comparar_opciones(listing_ids: list[str]):
    """Compara entre 2 y 4 alojamientos lado a lado (limpieza, ubicación, tranquilidad, anfitrión, fidelidad
    al anuncio, precio/calidad, precio y puntaje). Usala cuando el usuario dude entre opciones."""
    aspectos = {"Limpieza": ("puntaje_limpieza", "review_scores_cleanliness"),
                "Ubicación": ("puntaje_ubicacion", "review_scores_location"),
                "Tranquilidad": ("puntaje_tranquilidad", None),
                "Anfitrión": ("puntaje_anfitrion", "review_scores_communication"),
                "Fidelidad al anuncio": ("puntaje_fidelidad", "review_scores_accuracy")}
    filas = [f for f in (_fila(i) for i in listing_ids[:4]) if f is not None]
    if len(filas) < 2:
        return "Necesito al menos 2 alojamientos válidos para comparar.", {"tipo": "error"}
    comparacion = []
    for f in filas:
        valores = {}
        for nombre, (col_nlp, col_airbnb) in aspectos.items():
            v = f.get(col_nlp)
            if (v is None or pd.isna(v)) and col_airbnb:
                v = f.get(col_airbnb) * 2  # respaldo: subrating de Airbnb llevado a escala 10
            valores[nombre] = None if v is None or pd.isna(v) else round(float(v), 1)
        valores["Precio/calidad"] = round(float(f["c_precio_calidad"]) * 10, 1)
        comparacion.append({"id": str(int(f["id"])), "name": f["name"], "destino": f["destino"], "price": float(f["price"]),
                            "rating_airbnb_10": round(float(f["review_scores_rating"]) * 2, 1), "aspectos": valores})
    return json.dumps(comparacion, ensure_ascii=False), {"tipo": "comparacion", "opciones": comparacion}


@tool(response_format="content_and_artifact")
def clima_destino(destino: str, check_in: str, check_out: str):
    """Clima esperado en un destino para las fechas del viaje (temperaturas máximas/mínimas y lluvia) usando
    Open-Meteo: pronóstico si el viaje es en los próximos 14 días, si no referencia histórica del año anterior."""
    destino = destino.lower().strip()
    if destino not in DESTINOS:
        return f"Destino no disponible. Opciones: {', '.join(DESTINOS)}", {"tipo": "error"}
    try:
        datos = clima(destino, check_in, check_out)
    except Exception as e:
        return f"No pude consultar el clima ahora ({type(e).__name__}).", {"tipo": "error"}
    contenido = {k: v for k, v in datos.items() if k != "dias"}
    return json.dumps(contenido, ensure_ascii=False), {"tipo": "clima", **datos}


@tool(response_format="content_and_artifact")
def comparar_barrios(destino: str, huespedes: int = 2):
    """Compara los barrios/zonas de un destino: precio mediano por noche, rating promedio y cantidad de
    alojamientos para esa cantidad de huéspedes. Útil para recomendar dónde alojarse."""
    destino = destino.lower().strip()
    if destino not in DESTINOS:
        return f"Destino no disponible. Opciones: {', '.join(DESTINOS)}", {"tipo": "error"}
    pool = scoring.datos()["pool"]
    df = pool[(pool["destino"] == destino) & (pool["accommodates"] >= huespedes)]
    tabla = (df.groupby("barrio").agg(alojamientos=("id", "size"), precio_mediano=("price", "median"),
                                     rating_10=("review_scores_rating", lambda s: round(s.mean() * 2, 2)),
                                     ubicacion_10=("review_scores_location", lambda s: round(s.mean() * 2, 2)))
             .query("alojamientos >= 5").sort_values("rating_10", ascending=False).round(1).reset_index())
    top = tabla.head(10)
    return (json.dumps({"destino": destino, "barrios": top.to_dict("records")}, ensure_ascii=False),
            {"tipo": "barrios", "destino": destino, "tabla": top.to_dict("records")})


@tool(response_format="content_and_artifact")
def seleccionar_alojamiento(destino: str, listing_id: str, runtime: ToolRuntime = None):
    """Registra en el plan de viaje el alojamiento que eligió el usuario para un destino.
    Solo usala cuando el usuario confirme su elección."""
    thread = _thread(runtime)
    opciones = ULTIMAS_OPCIONES[thread].get(destino.lower(), [])
    lid = _id(listing_id)
    elegida = next((t for t in opciones if lid is not None and int(t["id"]) == lid), None)
    if elegida is None:
        return ("Ese alojamiento no está entre las últimas opciones mostradas para ese destino. "
                "Buscá opciones primero."), {"tipo": "error"}
    PLANES[thread][destino.lower()] = elegida
    return (f"Guardado: {elegida['name']} en {destino} ({elegida['noches']} noches, {elegida['total']} €).",
            {"tipo": "seleccion", "tarjeta": elegida})


@tool(response_format="content_and_artifact")
def resumen_viaje(runtime: ToolRuntime = None):
    """Devuelve el plan de viaje completo con los alojamientos elegidos, noches y costo total."""
    plan = PLANES[_thread(runtime)]
    if not plan:
        return "Todavía no hay alojamientos elegidos en el plan.", {"tipo": "plan", "tramos": []}
    tramos = sorted(plan.values(), key=lambda t: t["check_in"])
    total = round(sum(t["total"] for t in tramos), 2)
    contenido = {"tramos": [{k: t[k] for k in ["destino", "name", "check_in", "check_out", "noches", "total"]}
                            for t in tramos], "costo_total_eur": total}
    return json.dumps(contenido, ensure_ascii=False), {"tipo": "plan", "tramos": tramos, "total": total}


TOOLS = [buscar_opciones, detalle_alojamiento, consultar_resenas, comparar_opciones, clima_destino,
         comparar_barrios, seleccionar_alojamiento, resumen_viaje]

In [ ]:
%%writefile core/agent.py
"""Agente de viajes: create_agent + tools propias + memoria por sesión (checkpointer con thread_id)."""
from datetime import date

from langchain.agents import create_agent
from langchain.agents.middleware import ContextEditingMiddleware, ModelFallbackMiddleware
from langchain.agents.middleware.context_editing import ClearToolUsesEdit
from langchain_groq import ChatGroq
from langgraph.checkpoint.memory import InMemorySaver

from core.config import MODELO_BATCH
from core.nlp import llm_agente
from core.tools import TOOLS

SYSTEM_PROMPT = f"""Sos "Ruta Ibérica", un asesor de viajes que planifica estadías en Barcelona, Madrid y Mallorca
usando datos reales de Inside Airbnb (catálogo de alojamientos activos con reseñas analizadas por IA). Hoy es {date.today():%Y-%m-%d}.

Cómo trabajás:
1. Si falta información clave (destinos, fechas o cantidad de huéspedes), preguntala antes de buscar. El presupuesto es opcional.
2. Para cada destino del viaje llamá a buscar_opciones (una llamada por destino). Si no se indica año, usá la próxima fecha futura.
3. La interfaz ya muestra tarjetas con foto, precio, puntaje, mapa y gráficos: vos comentá en 3-5 frases por destino en qué
   se diferencian las opciones y por qué el modelo recomienda la suya. NO uses tablas ni repitas la ficha completa.
4. Cuando el usuario elija una opción, registrala con seleccionar_alojamiento. Al final ofrecé el resumen_viaje.
5. Si quiere estar cerca de un lugar (monumento, museo, playa, aeropuerto), pasalo en el parámetro cerca_de de buscar_opciones.
6. Preguntas puntuales sobre una opción ("¿es ruidoso?", "¿tiene ascensor?", "¿está limpio el baño?") → consultar_resenas
   con el id del alojamiento. Para un panorama general de una opción → detalle_alojamiento.
7. Si duda entre opciones → comparar_opciones con sus ids. Si pregunta por zonas → comparar_barrios.
8. Si pregunta por el clima o qué ropa llevar → clima_destino (podés ofrecerlo al mostrar opciones).
9. Los ids de los alojamientos son textos largos que vienen en los resultados de buscar_opciones: copialos
   exactos, como string, sin redondear ni inventar. Si una tool devuelve error, no la repitas con los mismos datos.

Reglas de evidencia:
- Solo afirmá datos que vengan de las tools. Si algo no está en los datos (desayuno, pileta, vistas), decí que no lo sabés.
- Los precios son en euros por noche según Inside Airbnb (junio 2026) y la disponibilidad es la del último relevamiento: aclaralo si preguntan.
- Si una tool devuelve error, explicá el problema en lenguaje simple y proponé una alternativa.
- Nunca muestres los ids al usuario: nombrá los alojamientos por su nombre o etiqueta (recomendada, más barata...).
Respondé en español neutro latinoamericano (sin "vosotros"), con tono cálido y concreto, en MÁXIMO 120 palabras y sin tablas."""

memoria = InMemorySaver()


def crear_agente(callbacks=None):
    """Agente con memoria por thread_id y dos middlewares pensados para el free tier de Groq:
    - ModelFallbackMiddleware: si el modelo principal agota su cuota por minuto, sigue con otro modelo (cada uno tiene su cuota).
    - ContextEditingMiddleware: limpia resultados viejos de tools para que el historial no crezca sin control
      (las búsquedas se conservan porque contienen los ids de los alojamientos)."""
    principal = llm_agente(callbacks=callbacks or [], max_retries=0)
    respaldos = [ChatGroq(model=MODELO_BATCH, temperature=0.2, reasoning_effort="low", max_retries=2),
                 ChatGroq(model="qwen/qwen3.8-27b", temperature=0.2, max_tokens=2048, max_retries=2)]
    return create_agent(
        model=principal,
        tools=TOOLS,
        system_prompt=SYSTEM_PROMPT,
        checkpointer=memoria,
        middleware=[
            ModelFallbackMiddleware(*respaldos),
            ContextEditingMiddleware(edits=[ClearToolUsesEdit(trigger=3000, keep=2, exclude_tools=("buscar_opciones",),
                                                              placeholder="[resultado anterior resumido]")]),
        ],
    )

In [ ]:
%%writefile core/enrich_runner.py
"""Corre el batch NLP sobre el catálogo en tandas, priorizando los alojamientos con más chances de mostrarse.

Uso:  python -m core.enrich_runner --limite 150 --modelo openai/gpt-oss-20b
Cada modelo de Groq tiene su propia cuota diaria gratuita: se puede repartir el trabajo entre modelos.
"""
import argparse
import time

import pandas as pd
from langchain_core.rate_limiters import InMemoryRateLimiter
from langchain_groq import ChatGroq

from core import scoring
from core.config import PROCESSED, asegurar_api_key
from core.metrics import MetricsTracker
from core.nlp import cargar_enriquecido, enriquecer, guardar_enriquecido, preparar_entradas


def prioridad(pool: pd.DataFrame) -> pd.DataFrame:
    """Ranking dentro de cada destino y capacidad: extremos de precio, mejor rating y mejor puntaje preliminar."""
    g = pool.groupby(["destino", "cap_bucket"])
    rangos = pd.concat([
        g["price"].rank(method="first"),
        g["price"].rank(method="first", ascending=False),
        g["rating_bayes"].rank(method="first", ascending=False),
        g["c_precio_calidad"].rank(method="first", ascending=False),
    ], axis=1)
    return pool.assign(prioridad=rangos.min(axis=1)).sort_values(["prioridad", "destino"])


def main() -> None:
    parser = argparse.ArgumentParser()
    parser.add_argument("--limite", type=int, default=100)
    parser.add_argument("--modelo", default="openai/gpt-oss-20b")
    parser.add_argument("--rps", type=float, default=0.1, help="requests por segundo (8K tokens/min en free tier)")
    args = parser.parse_args()
    asegurar_api_key()

    pool = prioridad(scoring.datos()["pool"])
    hechos = set(cargar_enriquecido()["listing_id"])
    pendientes = pool[~pool["id"].isin(hechos)].head(args.limite)
    resenas = pd.read_parquet(PROCESSED / "reviews_pool.parquet")
    entradas = preparar_entradas(pendientes, resenas[resenas["listing_id"].isin(pendientes["id"])])
    print(f"Pendientes totales: {len(pool) - len(hechos)} · esta tanda: {len(entradas)} con {args.modelo}")

    extra = {"reasoning_effort": "low"} if "gpt-oss" in args.modelo else {}
    llm = ChatGroq(model=args.modelo, temperature=0, max_tokens=900, max_retries=8, **extra,
                   rate_limiter=InMemoryRateLimiter(requests_per_second=args.rps, max_bucket_size=1))
    tracker = MetricsTracker()
    for i in range(0, len(entradas), 20):  # guarda cada 20 para no perder avance si se corta la cuota
        t0 = time.perf_counter()
        df = enriquecer(entradas[i:i + 20], callbacks=[tracker], max_concurrency=2, llm=llm)
        total = guardar_enriquecido(df)
        errores = int(df["error"].notna().sum()) if "error" in df else 0
        print(f"  tanda {i // 20 + 1}: {len(df) - errores} ok, {errores} errores, {time.perf_counter() - t0:.0f}s "
              f"· acumulado {len(total)} analizados")
        if errores == len(df):
            print("  Todas fallaron (¿cuota diaria agotada?). Corto acá.")
            break
    print(tracker.resumen(n_unidades=len(entradas)))


if __name__ == "__main__":
    main()

## 2 · Dataset: descarga y construcción del catálogo
Se descargan los CSV de Inside Airbnb (~420 MB) y se arma un catálogo curado de **500 alojamientos activos por destino**
(≥15 reseñas, reseña reciente, precio publicado), muestreado de forma estratificada por capacidad y rango de precio.

In [ ]:
from data.download_data import descargar
from core import data_prep
if not (data_prep.PROCESSED / "pool.parquet").exists():
    descargar()
    data_prep.construir()
else:
    print("Catálogo ya construido ✔")

In [ ]:
import pandas as pd
pd.set_option("display.max_colwidth", 80)
pool = pd.read_parquet("data/processed/pool.parquet")
cal = pd.read_parquet("data/processed/calendar_pool.parquet")
rev = pd.read_parquet("data/processed/reviews_pool.parquet")
print(f"Alojamientos: {len(pool)} · días de calendario: {len(cal):,} · reseñas para el batch: {len(rev):,}")
pool.groupby("destino").agg(alojamientos=("id", "size"), precio_mediano=("price", "median"),
                            rating_medio=("review_scores_rating", "mean"), capacidad_media=("accommodates", "mean"),
                            barrios=("barrio", "nunique")).round(2)

In [ ]:
from core.nlp import detectar_idioma
idiomas = rev["comments"].map(detectar_idioma).value_counts(normalize=True).round(3)
print("Idiomas detectados en las reseñas (heurística por stopwords):")
idiomas

## 3 · Pipeline LCEL: prompt | modelo | parser
Primer ejemplo mínimo con el operador `|`: prompt con reglas → LLM → `StrOutputParser` → transformación.

In [ ]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableLambda
from core.metrics import MetricsTracker
from core.nlp import llm_agente

tracker = MetricsTracker()   # callback que registra tokens, tiempo y costo de TODAS las llamadas
prompt = ChatPromptTemplate.from_messages([
    ("system", "Sos un asesor de viajes. Respondé en una sola frase y solo con información general conocida; "
               "si no sabés algo, decilo."),
    ("human", "¿Qué zona de {ciudad} conviene para un viajero que busca {perfil}?"),
])
cadena = prompt | llm_agente() | StrOutputParser() | RunnableLambda(lambda s: "🧭 " + s.strip())
print(cadena.invoke({"ciudad": "Madrid", "perfil": "museos y vida nocturna"}, config={"callbacks": [tracker]}))

## 4 · Salida estructurada con Pydantic: interpretar el pedido del viajero
`with_structured_output(SolicitudViaje)` convierte lenguaje natural en datos validados (tramos, fechas, huéspedes, presupuesto).

In [ ]:
from core.nlp import cadena_solicitud
solicitud = cadena_solicitud().invoke(
    "Somos 3, arrancamos en Barcelona el 10 de octubre, 3 noches, después Madrid 3 noches y terminamos con 4 en Mallorca. "
    "No más de 180 euros la noche, algo tranquilo.", config={"callbacks": [tracker]})
solicitud

## 5 · Procesamiento multidimensional con RunnableParallel
Sobre las reseñas de un mismo alojamiento corren **tres análisis en paralelo**:
1. `nlp`: LLM con salida estructurada `AnalisisResenas` (sentimiento, 5 puntajes por aspecto, pros, contras, perfiles, alerta, cita textual).
2. `senales`: detección léxica multilingüe de temas (ruido, transporte, playa, limpieza...).
3. `stats`: idiomas y volumen del texto.

In [ ]:
from core.nlp import cadena_analisis, preparar_entradas
ejemplo = preparar_entradas(pool[pool["destino"] == "madrid"].head(1), rev)[0]
print(ejemplo["nombre"], "\n", ejemplo["resenas"][:600], "...")
resultado = cadena_analisis().invoke(ejemplo, config={"callbacks": [tracker]})
print("\nSEÑALES:", resultado["senales"], "\nSTATS:", resultado["stats"])
resultado["nlp"]

## 6 · Procesamiento del dataset con `.batch()` → DataFrame enriquecido
Se analiza una muestra de alojamientos en paralelo (`max_concurrency`), con manejo de errores por registro
(`return_exceptions=True`) y un *rate limiter* para respetar la cuota gratuita (8K tokens/minuto).
El catálogo completo se procesa en tandas con `core/enrich_runner.py` y se cachea en `reviews_enriched.parquet`.

In [ ]:
import time
from core.nlp import enriquecer, guardar_enriquecido, cargar_enriquecido
tracker_batch = MetricsTracker()
ya = set(cargar_enriquecido()["listing_id"])
muestra = pool[~pool["id"].isin(ya)].groupby("destino").head(4)          # 12 alojamientos nuevos (4 por destino)
entradas = preparar_entradas(muestra, rev)
t0 = time.perf_counter()
df_batch = enriquecer(entradas, callbacks=[tracker_batch], max_concurrency=2)
print(f"{len(df_batch)} alojamientos en {time.perf_counter() - t0:.1f}s")
enriquecido = guardar_enriquecido(df_batch)
df_batch[["listing_id", "sentimiento_general", "puntaje_limpieza", "puntaje_tranquilidad", "pros", "contras", "resumen"]].head(8)

In [ ]:
metricas_batch = tracker_batch.resumen(n_unidades=len(entradas))
pd.Series(metricas_batch, name="batch de reseñas")

In [ ]:
# DataFrame enriquecido completo (muestra + tandas previas) unido al catálogo
enriquecido = cargar_enriquecido()
df_rico = pool.merge(enriquecido, left_on="id", right_on="listing_id")
print(f"Alojamientos con análisis NLP: {len(df_rico)} de {len(pool)}")
df_rico.groupby("destino").agg(n=("id", "size"), limpieza=("puntaje_limpieza", "mean"),
                               tranquilidad=("puntaje_tranquilidad", "mean"),
                               con_alerta=("alerta", lambda s: s.notna().sum())).round(2)

**Validación del análisis NLP.** Si el LLM lee bien las reseñas, sus puntajes por aspecto deberían correlacionar
con los sub-ratings numéricos que los huéspedes cargan en Airbnb (que el modelo **no** ve).

In [ ]:
pares = {"puntaje_limpieza": "review_scores_cleanliness", "puntaje_ubicacion": "review_scores_location",
         "puntaje_anfitrion": "review_scores_communication", "puntaje_fidelidad": "review_scores_accuracy"}
pd.DataFrame({
    "correlación de Spearman": {k: round(df_rico[k].rank().corr(df_rico[v].rank()), 3) for k, v in pares.items()},
    "n": {k: int(df_rico[[k, v]].dropna().shape[0]) for k, v in pares.items()},
})

## 7 · Modelo de puntaje (1 a 10) y selección de las 4 opciones
| Componente | Peso | Cómo se calcula |
|---|---|---|
| Calidad | 30 % | rating de Airbnb con **promedio bayesiano** (evita que un 5.0 con 3 reseñas gane) |
| Reseñas IA | 30 % | puntajes por aspecto + sentimiento del batch NLP (penaliza alertas) |
| Precio/calidad | 20 % | precio vs. mediana de su barrio y capacidad |
| Ubicación | 10 % | rating de ubicación + puntaje de ubicación NLP |
| Ajuste de capacidad | 10 % | penaliza alojamientos sobredimensionados para el grupo |

Filtros duros: **capacidad ≥ huéspedes**, **disponible todas las noches** (calendario) y estadía mínima compatible.

In [ ]:
from core import scoring
scoring.recargar()
cands = scoring.candidatos("barcelona", "2026-10-10", "2026-10-13", huespedes=2)
print(f"Candidatos disponibles: {len(cands)}")
pd.DataFrame([{"opción": et, "alojamiento": f["name"][:45], "€/noche": f["price"], "puntaje": f["puntaje"],
               "rating Airbnb /10": f["rating_airbnb_10"], **scoring.detalle_puntaje(f)}
              for et, f in scoring.elegir_cuatro(cands)])

## 8 · Tools propias del agente
| Tool | Operación real |
|---|---|
| `buscar_opciones` | filtra catálogo + calendario + capacidad (+ cercanía a un punto de interés) y devuelve las 4 opciones |
| `detalle_alojamiento` | análisis NLP de un alojamiento (lo calcula en vivo si falta) |
| `consultar_resenas` | RAG liviano: palabras clave multilingües → recuperación → respuesta con citas |
| `comparar_opciones` | comparación por aspecto (gráfico radar en la interfaz) |
| `clima_destino` | Open-Meteo: pronóstico o histórico para las fechas |
| `comparar_barrios` | KPIs por barrio |
| `seleccionar_alojamiento` | registra la elección en el plan de la sesión |
| `resumen_viaje` | plan final con costo total |

Las tools usan `response_format="content_and_artifact"`: el LLM recibe un resumen compacto (menos tokens) y la interfaz
recibe el *artifact* completo para dibujar tarjetas y gráficos.

In [ ]:
import json
from core import tools
contenido, artifact = tools.ejecutar_busqueda("madrid", "2026-10-13", "2026-10-16", 2, cerca_de="Museo del Prado")
print(json.dumps(json.loads(contenido)["opciones"], ensure_ascii=False, indent=1)[:1500])

In [ ]:
contenido, art = tools.clima_destino.func("mallorca", "2026-10-16", "2026-10-20")
print(contenido)

In [ ]:
id_recomendada = artifact["tarjetas"][0]["id"]
contenido, art = tools.consultar_resenas.func(id_recomendada, "¿Se escucha ruido de la calle a la noche?")
print(json.dumps(art, ensure_ascii=False, indent=1) if art.get("tipo") == "respuesta_resenas" else contenido)

## 9 · Memoria conversacional
**a) `RunnableWithMessageHistory`** sobre una cadena LCEL: el historial se guarda por `session_id`.

In [ ]:
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.prompts import MessagesPlaceholder
from langchain_core.runnables.history import RunnableWithMessageHistory

historiales = {}
def obtener_historial(session_id: str):
    return historiales.setdefault(session_id, InMemoryChatMessageHistory())

prompt_chat = ChatPromptTemplate.from_messages([
    ("system", "Sos Ruta Ibérica, asesor de viajes. Respondé breve."),
    MessagesPlaceholder("historial"),
    ("human", "{mensaje}"),
])
chat = RunnableWithMessageHistory(prompt_chat | llm_agente() | StrOutputParser(), obtener_historial,
                                  input_messages_key="mensaje", history_messages_key="historial")
cfg = {"configurable": {"session_id": "viajero-42"}, "callbacks": [tracker]}
print(chat.invoke({"mensaje": "Hola, me llamo Lucía y viajo con mi perro."}, config=cfg))
print(chat.invoke({"mensaje": "¿Cómo me llamo y con quién viajo?"}, config=cfg))

**b) El agente** usa un *checkpointer* de LangGraph: cada sesión de Chainlit es un `thread_id` distinto.

## 10 · Agente inteligente (`create_agent`)
El agente decide qué tools usar y en qué orden. Mostramos las decisiones (tool calls) de una conversación de varios turnos.

In [ ]:
from langchain_core.messages import AIMessage, ToolMessage
from core.agent import crear_agente
agente = crear_agente()
tracker_agente = MetricsTracker()
cfg_agente = {"configurable": {"thread_id": "demo-notebook"}, "callbacks": [tracker_agente]}

def conversar(texto):
    print(f"\n👤 {texto}")
    n0, t0 = len(tracker_agente.registros), time.perf_counter()
    for paso in agente.stream({"messages": [{"role": "user", "content": texto}]}, config=cfg_agente, stream_mode="updates"):
        for actualizacion in paso.values():
            for m in (actualizacion or {}).get("messages", []):
                if isinstance(m, AIMessage) and m.tool_calls:
                    for tc in m.tool_calls:
                        print(f"   🔧 decide usar {tc['name']}({json.dumps(tc['args'], ensure_ascii=False)})")
                elif isinstance(m, AIMessage) and m.content:
                    print(f"🤖 {m.content}")
    r = MetricsTracker(registros=tracker_agente.registros[n0:]).resumen(1)
    print(f"   ⚡ {time.perf_counter() - t0:.1f}s · {r['tokens_totales']} tokens · US$ {r['costo_total_usd']:.5f} (simulado)")

conversar("Hola! Somos 2 y queremos ir a Madrid del 13 al 16 de octubre, cerca del Museo del Prado, hasta 200 euros la noche.")

In [ ]:
conversar("¿La recomendada es ruidosa de noche?")

In [ ]:
conversar("Me quedo con la recomendada. ¿Qué clima vamos a tener? Y mostrame el plan.")

In [ ]:
conversar("¿Te acordás cuántos viajamos y cuál era nuestro presupuesto?")

## 11 · Métricas de funcionamiento y simulación económica
El uso real es **gratuito** (free tier de Groq). Para evaluar la viabilidad en producción se simula el costo con los
precios de lista de los mismos modelos y se compara con un proveedor premium (Claude Sonnet 5).

In [ ]:
from core.metrics import costo_usd
filas = {
    "Cadenas LCEL (secciones 3-5 y 9)": tracker.resumen(),
    "Batch de reseñas (sección 6)": tracker_batch.resumen(n_unidades=len(entradas)),
    "Agente (sección 10)": tracker_agente.resumen(n_unidades=4),
}
tabla = pd.DataFrame(filas).T[["llamadas", "tokens_entrada", "tokens_salida", "tokens_totales", "segundos_llm",
                                "costo_total_usd", "costo_promedio_usd", "costo_si_fuera_claude_sonnet_usd"]]
tabla.columns = ["llamadas", "tokens in", "tokens out", "tokens totales", "tiempo LLM (s)",
                 "costo total US$", "costo prom. por registro/consulta US$", "con Claude Sonnet 5 US$"]
tabla

In [ ]:
# Proyección: catálogo completo (1.500 alojamientos) + 1.000 consultas al mes
por_registro = tracker_batch.resumen(n_unidades=len(entradas))["costo_promedio_usd"]
por_consulta = tracker_agente.resumen(n_unidades=4)["costo_promedio_usd"]
pd.Series({
    "Analizar todo el catálogo (1.500 alojamientos)": round(por_registro * 1500, 4),
    "1.000 consultas de usuarios / mes": round(por_consulta * 1000, 4),
    "Total mensual estimado (US$)": round(por_registro * 1500 + por_consulta * 1000, 4),
}, name="US$ con precios de lista de Groq")

## 12 · Interfaz Chainlit
Archivos de la interfaz: `app.py`, componentes React propios (`public/elements/*.jsx`), estilos, tema y configuración.

In [ ]:
%%writefile app.py
"""Ruta Ibérica — interfaz Chainlit del agente de viajes (Barcelona · Madrid · Mallorca)."""
import asyncio
import json
import os
import time

import chainlit as cl
import groq
import plotly.graph_objects as go
from chainlit.input_widget import Slider
from langchain_core.messages import AIMessage, AIMessageChunk, ToolMessage

from core import tools
from core.agent import crear_agente
from core.config import DESTINOS
from core.lugares import LUGARES
from core.metrics import MetricsTracker
from core.scoring import PESOS


COLORES = {"recomendada": "#C8553D", "mejor_rateada": "#E9A23B", "mas_barata": "#2E8B6E", "mas_cara": "#5B4B8A"}
NOMBRES_TOOLS = {
    "buscar_opciones": "🔎 Buscando alojamientos",
    "detalle_alojamiento": "📝 Analizando reseñas",
    "comparar_barrios": "📊 Comparando barrios",
    "consultar_resenas": "🔍 Buscando en las reseñas",
    "comparar_opciones": "⚖️ Comparando opciones",
    "clima_destino": "🌤️ Consultando el clima",
    "seleccionar_alojamiento": "📌 Guardando elección",
    "resumen_viaje": "🧳 Armando el resumen",
}
SUGERENCIAS = [
    ("🗺️ Ruta completa", "Somos 2, queremos hacer Barcelona del 10 al 13 de octubre, Madrid del 13 al 16 y Mallorca del 16 al 20."),
    ("👨‍👩‍👧 Familia en Mallorca", "Vamos 4 personas (2 adultos y 2 chicos) a Mallorca del 2 al 9 de noviembre, hasta 250 € por noche."),
    ("🖼️ Cerca del Prado", "Busco algo para 1 persona en Madrid del 5 al 8 de diciembre, cerca del Museo del Prado. ¿Qué clima hace esos días?"),
    ("📍 ¿Dónde alojarme?", "¿Qué barrios de Barcelona me recomendás para 3 personas? Comparalos."),
]

BIENVENIDA = """
<div class="ri-hero">
  <h2>Hola, soy Ruta Ibérica 🧭</h2>
  <p>Te ayudo a planificar tu viaje por España eligiendo dónde dormir en cada destino, con datos reales de
  <b>Inside Airbnb</b> y miles de reseñas de huéspedes analizadas con IA.</p>
  <div class="ri-dest"><span>🏛️ Barcelona</span><span>🎨 Madrid</span><span>🏖️ Mallorca</span></div>
</div>
<div class="ri-pasos">
  <div class="ri-paso"><b>1 · Contame tu viaje</b>Destinos, fechas, cuántos viajan y, si querés, tu presupuesto por noche.</div>
  <div class="ri-paso"><b>2 · Te muestro 4 opciones</b>Recomendada por el modelo, mejor rateada, más barata y más cara, con puntaje de 1 a 10.</div>
  <div class="ri-paso"><b>3 · Elegís y armamos la ruta</b>Guardo tus elecciones y te doy el resumen con el costo total.</div>
</div>

También podés preguntarme **qué dicen las reseñas** de una opción (*"¿es ruidoso de noche?"*), **comparar** opciones,
consultar **el clima** de tus fechas o pedir alojamientos **cerca de** un lugar. Con el ícono ⚙️ ajustás qué pesa más
en el puntaje del modelo.

Probá con una de estas ideas 👇
"""


def mapa_opciones(tarjetas: list[dict], destino: str, cerca_de: str | None = None) -> go.Figure:
    fig = go.Figure()
    for t in tarjetas:
        fig.add_trace(go.Scattermapbox(
            lat=[t["latitude"]], lon=[t["longitude"]], mode="markers", name=t["titulo_etiqueta"],
            marker=dict(size=17, color=COLORES.get(t["etiqueta"], "#1F5673")),
            text=f"{t['name']}<br>{t['price']:.0f} €/noche · {t['puntaje']}/10", hoverinfo="text",
        ))
    lats, lons = [t["latitude"] for t in tarjetas], [t["longitude"] for t in tarjetas]
    if cerca_de:
        lat_l, lon_l = LUGARES[destino][cerca_de]
        fig.add_trace(go.Scattermapbox(lat=[lat_l], lon=[lon_l], mode="markers+text", name=f"📍 {cerca_de}",
                                       marker=dict(size=13, color="#111827"), text=[cerca_de],
                                       textposition="top center", hoverinfo="text"))
        lats, lons = lats + [lat_l], lons + [lon_l]
    extension = max(max(lats) - min(lats), max(lons) - min(lons))
    zoom = 13 if extension < 0.015 else 12 if extension < 0.04 else 11 if extension < 0.12 else 8.3
    fig.update_layout(
        mapbox=dict(style="open-street-map", center=dict(lat=sum(lats) / len(lats), lon=sum(lons) / len(lons)), zoom=zoom),
        margin=dict(l=0, r=0, t=0, b=0), height=340, legend=dict(orientation="h", y=-0.05),
    )
    return fig


def grafico_barrios(tabla: list[dict], destino: str) -> go.Figure:
    tabla = sorted(tabla, key=lambda r: r["precio_mediano"])
    fig = go.Figure(go.Bar(
        x=[r["precio_mediano"] for r in tabla], y=[r["barrio"] for r in tabla], orientation="h",
        marker=dict(color=[r["rating_10"] for r in tabla], colorscale=[[0, "#E9A23B"], [1, "#1F5673"]],
                    colorbar=dict(title="Rating /10")),
        text=[f"{r['precio_mediano']:.0f} € · ⭐ {r['rating_10']}" for r in tabla], textposition="auto",
    ))
    fig.update_layout(title=f"Precio mediano por noche en {DESTINOS[destino]['nombre']}", xaxis_title="€ / noche",
                      margin=dict(l=10, r=10, t=40, b=10), height=380)
    return fig


def mapa_ruta(tramos: list[dict]) -> go.Figure:
    fig = go.Figure(go.Scattermapbox(
        lat=[t["latitude"] for t in tramos], lon=[t["longitude"] for t in tramos], mode="lines+markers+text",
        line=dict(width=3, color="#C8553D"), marker=dict(size=14, color="#1F5673"),
        text=[f"{i + 1}. {t['destino'].title()}" for i, t in enumerate(tramos)], textposition="top right",
    ))
    fig.update_layout(mapbox=dict(style="open-street-map", center=dict(lat=40.3, lon=0.5), zoom=4.6),
                      margin=dict(l=0, r=0, t=0, b=0), height=300, showlegend=False)
    return fig


def radar_opciones(opciones: list[dict]) -> go.Figure:
    paleta = ["#C8553D", "#1F5673", "#2E8B6E", "#D48E1F"]
    fig = go.Figure()
    for color, o in zip(paleta, opciones):
        ejes = [k for k, v in o["aspectos"].items() if v is not None]
        valores = [o["aspectos"][k] for k in ejes]
        fig.add_trace(go.Scatterpolar(r=valores + valores[:1], theta=ejes + ejes[:1], fill="toself", opacity=0.55,
                                      name=f"{o['name'][:32]} · {o['price']:.0f} €", line=dict(color=color)))
    fig.update_layout(polar=dict(radialaxis=dict(range=[0, 10], tickvals=[2, 4, 6, 8, 10])),
                      legend=dict(orientation="h", y=-0.12), margin=dict(l=40, r=40, t=40, b=30), height=440,
                      title="Comparación por aspecto (1-10)")
    return fig


def grafico_clima(c: dict) -> go.Figure:
    fechas = [f"{d['fecha'][8:10]}/{d['fecha'][5:7]}" for d in c["dias"]]
    fig = go.Figure()
    fig.add_bar(x=fechas, y=[d["lluvia"] for d in c["dias"]], name="Lluvia (mm)", marker_color="#2B8FB5",
                opacity=0.45, yaxis="y2")
    fig.add_scatter(x=fechas, y=[d["max"] for d in c["dias"]], name="Máx °C", mode="lines+markers",
                    line=dict(color="#C8553D", width=3))
    fig.add_scatter(x=fechas, y=[d["min"] for d in c["dias"]], name="Mín °C", mode="lines+markers",
                    line=dict(color="#1F5673", width=3))
    fig.update_layout(title=dict(text=f"🌤️ {DESTINOS[c['destino']]['nombre']}: {c['max_promedio']}° / {c['min_promedio']}° · "
                                      f"{c['dias_con_lluvia']} día(s) con lluvia<br><sup>Fuente: Open-Meteo, {c['fuente']}</sup>"),
                      xaxis=dict(type="category"), yaxis=dict(title="°C"), yaxis2=dict(title="mm", overlaying="y", side="right", showgrid=False),
                      legend=dict(orientation="h", y=-0.2), margin=dict(l=10, r=10, t=50, b=10), height=320)
    return fig


async def renderizar(artifact: dict | None) -> None:
    """Convierte el artifact de cada tool en elementos visuales."""
    if not artifact:
        return
    tipo = artifact.get("tipo")
    if tipo == "opciones":
        destino = artifact["destino"].lower()
        t0 = artifact["tarjetas"][0]
        titulo = f"{DESTINOS[destino]['emoji']} {DESTINOS[destino]['nombre']} · {t0['check_in']} → {t0['check_out']}"
        await cl.Message(
            content=f"### {titulo}",
            elements=[
                cl.CustomElement(name="OpcionesAlojamiento", display="inline",
                                 props={"tarjetas": artifact["tarjetas"], "titulo": titulo,
                                        "candidatos": artifact["candidatos"], "huespedes": artifact["huespedes"],
                                        "cerca_de": artifact.get("cerca_de"), "relevamiento": artifact["relevamiento"]}),
                cl.Plotly(name=f"mapa_{destino}", figure=mapa_opciones(artifact["tarjetas"], destino, artifact.get("cerca_de")), display="inline"),
            ],
        ).send()
    elif tipo == "barrios":
        await cl.Message(content="", elements=[
            cl.Plotly(name="barrios", figure=grafico_barrios(artifact["tabla"], artifact["destino"]), display="inline")
        ]).send()
    elif tipo == "respuesta_resenas":
        await cl.Message(content="", elements=[
            cl.CustomElement(name="RespuestaResenas", display="inline",
                             props={k: v for k, v in artifact.items() if k != "tipo"})
        ]).send()
    elif tipo == "comparacion":
        await cl.Message(content="", elements=[
            cl.Plotly(name="comparacion", figure=radar_opciones(artifact["opciones"]), display="inline")
        ]).send()
    elif tipo == "clima":
        await cl.Message(content="", elements=[
            cl.Plotly(name=f"clima_{artifact['destino']}", figure=grafico_clima(artifact), display="inline")
        ]).send()
    elif tipo == "plan" and artifact.get("tramos"):
        await cl.Message(content="", elements=[
            cl.CustomElement(name="PlanViaje", display="inline",
                             props={"tramos": artifact["tramos"], "total": artifact["total"]}),
            cl.Plotly(name="ruta", figure=mapa_ruta(artifact["tramos"]), display="inline"),
        ]).send()


def acciones_sugeridas() -> list[cl.Action]:
    return [cl.Action(name="sugerencia", payload={"texto": texto}, label=etiqueta) for etiqueta, texto in SUGERENCIAS]


NOMBRES_PESOS = {
    "calidad": "⭐ Calidad (rating de Airbnb)",
    "resenas_nlp": "🧠 Reseñas analizadas con IA",
    "precio_calidad": "💶 Relación precio/calidad",
    "ubicacion": "📍 Ubicación",
    "ajuste_capacidad": "👥 Ajuste al tamaño del grupo",
}


@cl.on_chat_start
async def inicio():
    tracker = MetricsTracker()
    cl.user_session.set("tracker", tracker)
    cl.user_session.set("consultas", 0)
    cl.user_session.set("pesos", {k: v * 10 for k, v in PESOS.items()})
    await cl.ChatSettings([
        Slider(id=k, label=etiqueta, initial=PESOS[k] * 10, min=0, max=10, step=0.5,
               description="Cuánto pesa este componente en el puntaje de 1 a 10 del modelo")
        for k, etiqueta in NOMBRES_PESOS.items()
    ]).send()
    await cl.Message(content=BIENVENIDA, actions=acciones_sugeridas()).send()
    if not os.getenv("GROQ_API_KEY"):
        await cl.ErrorMessage(content="🔑 Falta configurar GROQ_API_KEY (archivo .env o variable de entorno). "
                                      "Podés ver las opciones, pero el asistente no puede responder todavía.").send()
        return
    cl.user_session.set("agente", crear_agente())


@cl.on_settings_update
async def on_settings(settings: dict):
    """Al mover los sliders se recalculan en vivo las últimas búsquedas, sin volver a llamar al agente."""
    pesos = {k: float(settings.get(k, PESOS[k] * 10)) for k in PESOS}
    cl.user_session.set("pesos", pesos)
    thread = cl.context.session.id
    busquedas = list(tools.ULTIMAS_BUSQUEDAS[thread].values())
    total = sum(pesos.values()) or 1
    resumen = " · ".join(f"{NOMBRES_PESOS[k].split(' ', 1)[1]} {v / total:.0%}" for k, v in pesos.items())
    if not busquedas:
        await cl.Message(content=f"⚙️ Pesos del modelo actualizados: {resumen}. Se aplican a tu próxima búsqueda.").send()
        return
    await cl.Message(content=f"⚙️ Pesos actualizados ({resumen}). Recalculo tus opciones…").send()
    for b in busquedas:
        _, artifact = await asyncio.to_thread(tools.ejecutar_busqueda, **b, pesos=pesos, thread=thread,
                                              callbacks=[cl.user_session.get("tracker")])
        await renderizar(artifact)


@cl.action_callback("sugerencia")
async def on_sugerencia(action: cl.Action):
    texto = action.payload["texto"]
    await cl.Message(content=texto, type="user_message").send()
    await responder(texto)


@cl.action_callback("metricas")
async def on_metricas(action: cl.Action):
    tracker: MetricsTracker = cl.user_session.get("tracker")
    consultas = cl.user_session.get("consultas") or 1
    resumen = tracker.resumen(n_unidades=consultas) | {"consultas": consultas,
                                                       "segundos": round(sum(r.segundos for r in tracker.registros), 1),
                                                       "titulo": "📊 Métricas acumuladas de la sesión"}
    await cl.Message(content="", elements=[cl.CustomElement(name="Metricas", props=resumen, display="inline")]).send()


@cl.on_message
async def on_message(message: cl.Message):
    await responder(message.content)


async def responder(texto: str) -> None:
    agente = cl.user_session.get("agente")
    if agente is None:
        await cl.ErrorMessage(content="🔑 El asistente no está configurado: falta GROQ_API_KEY.").send()
        return
    tracker: MetricsTracker = cl.user_session.get("tracker")
    config = {"configurable": {"thread_id": cl.context.session.id, "pesos": cl.user_session.get("pesos")},
              "callbacks": [tracker]}
    n_previos = len(tracker.registros)
    inicio = time.perf_counter()
    respuesta = cl.Message(content="")
    pasos: dict[str, cl.Step] = {}

    try:
        async for modo, dato in agente.astream({"messages": [{"role": "user", "content": texto}]},
                                               config=config, stream_mode=["messages", "updates"]):
            if modo == "messages":
                chunk, meta = dato
                if isinstance(chunk, AIMessageChunk) and meta.get("langgraph_node") == "model" and isinstance(chunk.content, str) and chunk.content:
                    await respuesta.stream_token(chunk.content)
                continue
            for actualizacion in dato.values():
                for msg in (actualizacion or {}).get("messages", []):
                    if isinstance(msg, AIMessage) and msg.tool_calls:
                        for llamada in msg.tool_calls:
                            paso = cl.Step(name=NOMBRES_TOOLS.get(llamada["name"], llamada["name"]), type="tool")
                            paso.input = json.dumps(llamada["args"], ensure_ascii=False, indent=1)
                            await paso.send()
                            pasos[llamada["id"]] = paso
                    elif isinstance(msg, ToolMessage):
                        paso = pasos.get(msg.tool_call_id)
                        if paso:
                            paso.output = str(msg.content)[:2000]
                            await paso.update()
                        if respuesta.content:  # cerrar el texto previo para que las tarjetas queden en orden
                            await respuesta.send()
                            respuesta = cl.Message(content="")
                        await renderizar(msg.artifact)
    except groq.RateLimitError:
        await cl.ErrorMessage(content="⏳ Alcanzamos el límite gratuito de consultas por minuto de Groq. "
                                      "Esperá unos segundos y volvé a intentarlo.").send()
        return
    except groq.APIConnectionError:
        await cl.ErrorMessage(content="🌐 No pude conectarme con el modelo. Revisá tu conexión e intentá de nuevo.").send()
        return
    except Exception as e:  # cualquier otro error se informa sin romper la sesión
        await cl.ErrorMessage(content=f"😕 Algo salió mal procesando tu pedido: {type(e).__name__}. Probá reformularlo.").send()
        return

    consultas = (cl.user_session.get("consultas") or 0) + 1
    cl.user_session.set("consultas", consultas)
    nuevos = MetricsTracker(registros=tracker.registros[n_previos:]).resumen(n_unidades=1)
    segundos = time.perf_counter() - inicio
    pie = (f"\n\n<span class='ri-metricas'>⚡ {segundos:.1f} s · {nuevos['tokens_totales']:,} tokens "
           f"({nuevos['tokens_entrada']:,} entrada / {nuevos['tokens_salida']:,} salida) · "
           f"US$ {nuevos['costo_total_usd']:.4f} simulado</span>").replace(",", ".")
    await respuesta.stream_token(pie)
    respuesta.actions = [cl.Action(name="metricas", payload={}, label="📊 Métricas de la sesión")]
    await respuesta.send()

In [ ]:
%%writefile public/elements/OpcionesAlojamiento.jsx
import { Star, Users, MapPin, ExternalLink, Check, AlertTriangle, BedDouble, Quote, Award, Navigation, Info } from "lucide-react"
import { useState } from "react"

const NOMBRES_PUNTAJE = {
  calidad: "Calidad (rating)",
  resenas_nlp: "Reseñas (IA)",
  precio_calidad: "Precio/calidad",
  ubicacion: "Ubicación",
  ajuste_capacidad: "Ajuste capacidad",
}

const euros = (n) => new Intl.NumberFormat("es-ES", { style: "currency", currency: "EUR", maximumFractionDigits: 0 }).format(n)

function Anillo({ valor }) {
  const r = 22, c = 2 * Math.PI * r, pct = Math.max(0, Math.min(1, (valor - 1) / 9))
  const color = valor >= 8.5 ? "#2E8B6E" : valor >= 7 ? "#E9A23B" : "#C8553D"
  return (
    <div className="ri-anillo" title="Puntaje Ruta Ibérica (1 a 10)">
      <svg viewBox="0 0 56 56">
        <circle cx="28" cy="28" r={r} className="ri-anillo-fondo" strokeWidth="5" fill="none" />
        <circle cx="28" cy="28" r={r} stroke={color} strokeWidth="5" fill="none" strokeLinecap="round"
          strokeDasharray={`${c * pct} ${c}`} transform="rotate(-90 28 28)" />
      </svg>
      <span>{valor?.toFixed(1)}</span>
    </div>
  )
}

function Tarjeta({ t }) {
  const [abierto, setAbierto] = useState(false)
  const foto = t.picture_url ? `${t.picture_url}?im_w=720` : null
  return (
    <div className={`ri-card ri-${t.etiqueta}`}>
      <div className="ri-foto">
        {foto && <img src={foto} alt={t.name} onError={(e) => { e.currentTarget.style.display = "none" }} />}
        <span className="ri-chip" title={t.titulo_etiqueta}>{t.etiqueta_corta || t.titulo_etiqueta}</span>
        {t.host_is_superhost === "t" && <span className="ri-superhost"><Award size={12} /> Superhost</span>}
      </div>

      <div className="ri-cuerpo">
        <div className="ri-fila-titulo">
          <div style={{ flex: 1, minWidth: 0 }}>
            <div className="ri-nombre">{t.name}</div>
            <div className="ri-sub"><MapPin size={12} /> {t.barrio}</div>
          </div>
          <Anillo valor={t.puntaje} />
        </div>

        <div className="ri-datos">
          <span><Users size={12} /> hasta {t.accommodates}</span>
          {t.bedrooms ? <span><BedDouble size={12} /> {t.bedrooms} dorm.</span> : null}
          <span><Star size={12} /> {t.rating_airbnb_10}/10 · {t.number_of_reviews} reseñas</span>
        </div>

        {(t.cercanias || []).length > 0 && (
          <div className="ri-cerca">
            {t.cercanias.map((c) => (
              <span key={c.lugar} title={`${c.lugar}: ${c.texto}`}><Navigation size={11} /><em>{c.lugar}</em><b>{c.texto}</b></span>
            ))}
          </div>
        )}

        <div className="ri-precio">
          <div><b>{euros(t.price)}</b><small> / noche</small></div>
          <small>{t.noches} noches · <b>{euros(t.total)}</b></small>
        </div>

        {t.resumen && <p className="ri-resumen">{t.resumen}</p>}
        {!t.tiene_nlp && <p className="ri-sub">Análisis de reseñas pendiente: puntaje calculado con los ratings de Airbnb.</p>}

        {t.alerta && <div className="ri-alerta"><AlertTriangle size={14} /> {t.alerta}</div>}

        <button className="ri-link" onClick={() => setAbierto(!abierto)}>
          {abierto ? "Ocultar detalle" : "Pros, contras y por qué este puntaje"}
        </button>

        {abierto && (
          <div className="ri-detalle">
            {(t.pros || []).map((p, i) => <div key={"p" + i}>✅ {p}</div>)}
            {(t.contras || []).map((p, i) => <div key={"c" + i}>⚠️ {p}</div>)}
            {t.cita_textual && <div className="ri-cita"><Quote size={12} /> “{t.cita_textual}”</div>}
            {Object.entries(t.detalle_puntaje || {}).map(([k, v]) => (
              <div key={k} className="ri-barra">
                <span>{NOMBRES_PUNTAJE[k] || k}</span>
                <div><i style={{ width: `${v * 10}%` }} /></div>
                <b>{v}</b>
              </div>
            ))}
            {(t.ideal_para || []).length > 0 && (
              <div className="ri-tags">{t.ideal_para.map((p) => <span key={p}>{p}</span>)}</div>
            )}
          </div>
        )}

        <div className="ri-botones">
          <button className="ri-btn-primario"
            onClick={() => sendUserMessage(`Elijo "${t.name}" (id ${t.id}) para ${t.destino}.`)}>
            <Check size={16} /> Elegir esta
          </button>
          <a className="ri-btn-secundario" href={t.listing_url} target="_blank" rel="noreferrer" title="Ver en Airbnb">
            <ExternalLink size={16} />
          </a>
        </div>
      </div>
    </div>
  )
}

export default function OpcionesAlojamiento() {
  const tarjetas = props.tarjetas || []
  return (
    <div className="ri-opciones">
      <div className="ri-sub">
        {props.candidatos} alojamientos disponibles evaluados para {props.huespedes} huésped{props.huespedes > 1 ? "es" : ""}
        {props.cerca_de ? ` · cerca de ${props.cerca_de}` : ""}
      </div>
      <div className="ri-grilla">
        {tarjetas.map((t) => <Tarjeta key={t.id} t={t} />)}
      </div>
      <div className="ri-aviso">
        <Info size={13} /> Disponibilidad y precios según el relevamiento de Inside Airbnb del {props.relevamiento}.
        Confirmá en Airbnb antes de reservar.
      </div>
    </div>
  )
}

In [ ]:
%%writefile public/elements/PlanViaje.jsx
import { MapPin, Moon, ExternalLink } from "lucide-react"

const euros = (n) => new Intl.NumberFormat("es-ES", { style: "currency", currency: "EUR", maximumFractionDigits: 0 }).format(n)
const fecha = (s) => new Date(s + "T12:00:00").toLocaleDateString("es-ES", { day: "numeric", month: "short" })
const EMOJI = { barcelona: "🏛️", madrid: "🎨", mallorca: "🏖️" }
const titulo = (d) => d.charAt(0).toUpperCase() + d.slice(1)

export default function PlanViaje() {
  const tramos = props.tramos || []
  const noches = tramos.reduce((a, t) => a + t.noches, 0)
  return (
    <div className="ri-plan">
      <div className="ri-plan-cabecera">
        <div>
          <small>TU RUTA</small>
          <h3>{tramos.map((t) => `${EMOJI[t.destino] || "📍"} ${titulo(t.destino)}`).join("  →  ")}</h3>
        </div>
        <div style={{ textAlign: "right" }}>
          <small>{noches} noches · alojamiento</small>
          <h3>{euros(props.total)}</h3>
        </div>
      </div>
      <div className="ri-plan-cuerpo">
        {tramos.map((t, i) => (
          <div key={t.id} className="ri-tramo">
            <div className="ri-num">{i + 1}</div>
            <img src={`${t.picture_url}?im_w=320`} onError={(e) => { e.currentTarget.style.visibility = "hidden" }} />
            <div style={{ flex: 1, minWidth: 0 }}>
              <div className="ri-nombre">{t.name}</div>
              <div className="ri-datos">
                <span><MapPin size={12} /> {t.barrio}, {titulo(t.destino)}</span>
                <span><Moon size={12} /> {fecha(t.check_in)} → {fecha(t.check_out)} ({t.noches} noches)</span>
                <span>Puntaje {t.puntaje}/10</span>
              </div>
            </div>
            <div style={{ textAlign: "right" }}>
              <b>{euros(t.total)}</b>
              <a href={t.listing_url} target="_blank" rel="noreferrer" className="ri-reservar">
                Ver en Airbnb <ExternalLink size={12} />
              </a>
            </div>
          </div>
        ))}
      </div>
    </div>
  )
}

In [ ]:
%%writefile public/elements/Metricas.jsx
import { Cpu, Timer, Coins, ArrowDownToLine, ArrowUpFromLine, Calculator } from "lucide-react"

const fmt = (n) => new Intl.NumberFormat("es-ES").format(n)

function Item({ icono, etiqueta, valor }) {
  return (
    <div className="ri-kpi">
      {icono}
      <div>
        <small>{etiqueta}</small>
        <b>{valor}</b>
      </div>
    </div>
  )
}

export default function Metricas() {
  const m = props
  return (
    <div className="ri-opciones">
      <div className="ri-sub">{m.titulo}</div>
      <div className="ri-kpis">
        <Item icono={<ArrowDownToLine size={18} />} etiqueta="Tokens de entrada" valor={fmt(m.tokens_entrada)} />
        <Item icono={<ArrowUpFromLine size={18} />} etiqueta="Tokens de salida" valor={fmt(m.tokens_salida)} />
        <Item icono={<Cpu size={18} />} etiqueta="Tokens totales" valor={fmt(m.tokens_totales)} />
        <Item icono={<Timer size={18} />} etiqueta="Tiempo de LLM" valor={`${m.segundos} s`} />
        <Item icono={<Coins size={18} />} etiqueta="Costo simulado total" valor={`US$ ${m.costo_total_usd.toFixed(4)}`} />
        <Item icono={<Calculator size={18} />} etiqueta="Promedio por consulta" valor={`US$ ${m.costo_promedio_usd.toFixed(5)}`} />
      </div>
      <div className="ri-sub">
        {m.consultas} consulta(s) · {m.llamadas} llamadas al LLM · costo real US$ 0 (free tier de Groq) ·
        con Claude Sonnet 5 costaría US$ {m.costo_si_fuera_claude_sonnet_usd}
      </div>
    </div>
  )
}

In [ ]:
%%writefile public/elements/RespuestaResenas.jsx
import { MessageSquareQuote, Search } from "lucide-react"

const COLORES = { "sí": "#2E8B6E", "no": "#C8553D", "mixto": "#D48E1F", "sin información": "#6B7280" }

export default function RespuestaResenas() {
  const color = COLORES[props.veredicto] || "#6B7280"
  return (
    <div className="ri-qa" style={{ borderLeftColor: color }}>
      <div className="ri-qa-cabecera">
        <span className="ri-qa-veredicto" style={{ background: color }}>{props.veredicto.toUpperCase()}</span>
        <span className="ri-sub"><Search size={12} /> “{props.pregunta}” · {props.nombre}</span>
      </div>
      <p className="ri-resumen">{props.respuesta}</p>
      {(props.evidencia || []).map((c, i) => (
        <div key={i} className="ri-cita"><MessageSquareQuote size={13} /> “{c.texto}” <small>— reseña del {c.fecha}</small></div>
      ))}
      <div className="ri-sub">
        {props.n_relevantes} de {props.n_total} reseñas mencionan el tema · búsqueda multilingüe:{" "}
        {(props.palabras || []).slice(0, 8).join(", ")}
      </div>
    </div>
  )
}

In [ ]:
%%writefile public/custom.css
/* Identidad visual Ruta Ibérica: terracota + azul mediterráneo + arena */
h1, h2, h3 { font-family: 'Fraunces', serif !important; letter-spacing: -0.01em; }

.ri-hero {
  border-radius: 1.2rem;
  padding: 1.4rem 1.5rem;
  color: #fff;
  background:
    radial-gradient(circle at 85% 20%, rgba(233, 162, 59, .55), transparent 45%),
    linear-gradient(120deg, #1F5673 0%, #2B6E8F 45%, #C8553D 100%);
  box-shadow: 0 10px 30px -12px rgba(31, 86, 115, .6);
}
.ri-hero h2 { margin: 0 0 .3rem; font-size: 1.7rem; color: #fff; }
.ri-hero p { margin: 0; opacity: .92; }
.ri-dest { display: flex; gap: .6rem; flex-wrap: wrap; margin-top: .9rem; }
.ri-dest span {
  background: rgba(255, 255, 255, .16); border: 1px solid rgba(255, 255, 255, .3);
  padding: .3rem .8rem; border-radius: 999px; font-size: .85rem; backdrop-filter: blur(4px);
}
.ri-pasos { display: grid; grid-template-columns: repeat(auto-fit, minmax(170px, 1fr)); gap: .6rem; margin-top: .8rem; }
.ri-paso { border: 1px solid hsl(var(--border)); border-radius: .9rem; padding: .7rem .8rem; background: hsl(var(--card)); font-size: .88rem; }
.ri-paso b { display: block; font-family: 'Fraunces', serif; font-size: 1rem; margin-bottom: .15rem; }
.ri-metricas { font-size: .75rem; opacity: .7; }

/* botones de acciones (preguntas sugeridas) con estilo de chip */
button[id^="action-"] {
  border-radius: 999px !important;
  border: 1px solid hsl(var(--primary) / .45) !important;
}

/* ---------- tarjetas de alojamiento ---------- */
.ri-opciones { width: 100%; display: flex; flex-direction: column; gap: .6rem; }
.ri-grilla { display: grid; grid-template-columns: repeat(auto-fit, minmax(min(100%, 190px), 1fr)); gap: .7rem; }
.ri-card {
  --ri-color: #1F5673;
  min-width: 0; overflow-wrap: anywhere;
  display: flex; flex-direction: column; overflow: hidden; border-radius: 1.1rem;
  background: hsl(var(--card)); color: hsl(var(--card-foreground));
  border: 1px solid hsl(var(--border)); box-shadow: 0 6px 18px -10px rgba(0,0,0,.35);
  transition: transform .15s ease, box-shadow .15s ease;
}
.ri-card:hover { transform: translateY(-3px); box-shadow: 0 14px 28px -12px rgba(0,0,0,.45); }
.ri-recomendada { --ri-color: #C8553D; border: 2px solid #C8553D; }
.ri-mejor_rateada { --ri-color: #D48E1F; }
.ri-mas_barata { --ri-color: #2E8B6E; }
.ri-mas_cara { --ri-color: #5B4B8A; }
.ri-foto { position: relative; height: 160px; background: linear-gradient(135deg, #F4E9D8, #C8553D66); }
.ri-foto img { width: 100%; height: 100%; object-fit: cover; display: block; }
.ri-chip {
  position: absolute; top: .55rem; left: .55rem; background: var(--ri-color); color: #fff;
  font-size: .72rem; font-weight: 600; padding: .25rem .65rem; border-radius: 999px; box-shadow: 0 2px 6px rgba(0,0,0,.25);
}
.ri-superhost {
  position: absolute; top: .55rem; right: .55rem; background: rgba(255,255,255,.92); color: #222;
  font-size: .68rem; padding: .15rem .5rem; border-radius: 999px; display: flex; align-items: center; gap: .25rem;
}
.ri-cuerpo { padding: .8rem; display: flex; flex-direction: column; gap: .5rem; flex: 1; }
.ri-fila-titulo { display: flex; gap: .6rem; align-items: flex-start; }
.ri-nombre { font-weight: 600; line-height: 1.25; overflow: hidden; display: -webkit-box; -webkit-line-clamp: 2; -webkit-box-orient: vertical; }
.ri-sub { font-size: .78rem; color: hsl(var(--muted-foreground)); display: flex; align-items: center; gap: .3rem; flex-wrap: wrap; }
.ri-datos { display: flex; flex-wrap: wrap; gap: .25rem .8rem; font-size: .76rem; color: hsl(var(--muted-foreground)); }
.ri-datos span { display: inline-flex; align-items: center; gap: .25rem; }
.ri-precio { display: flex; justify-content: space-between; align-items: baseline; gap: .5rem; flex-wrap: wrap; }
.ri-precio b { font-size: 1.25rem; }
.ri-precio small { font-size: .76rem; color: hsl(var(--muted-foreground)); }
.ri-precio small b { font-size: .85rem; color: hsl(var(--foreground)); }
.ri-resumen { font-size: .86rem; line-height: 1.35; margin: 0; }
.ri-alerta { font-size: .76rem; background: rgba(220, 38, 38, .12); color: #dc2626; border-radius: .6rem; padding: .45rem .55rem; display: flex; gap: .35rem; }
.ri-link { background: none; border: 0; padding: 0; text-align: left; font-size: .76rem; text-decoration: underline; color: hsl(var(--muted-foreground)); cursor: pointer; }
.ri-detalle { display: flex; flex-direction: column; gap: .3rem; font-size: .78rem; }
.ri-cita { display: flex; gap: .4rem; align-items: flex-start; font-style: italic; background: hsl(var(--muted)); border-radius: .6rem; padding: .45rem .55rem; }
.ri-barra { display: grid; grid-template-columns: 7.2rem 1fr 1.8rem; align-items: center; gap: .4rem; }
.ri-barra span { color: hsl(var(--muted-foreground)); }
.ri-barra div { height: 6px; border-radius: 99px; background: hsl(var(--muted)); overflow: hidden; }
.ri-barra i { display: block; height: 100%; background: linear-gradient(90deg, #1F5673, #2B8FB5); }
.ri-barra b { text-align: right; }
.ri-tags { display: flex; flex-wrap: wrap; gap: .3rem; }
.ri-tags span { background: rgba(31, 86, 115, .12); color: #2B8FB5; border-radius: 99px; padding: .1rem .5rem; }
.ri-botones { margin-top: auto; display: flex; gap: .45rem; padding-top: .3rem; }
.ri-btn-primario {
  flex: 1; display: inline-flex; justify-content: center; align-items: center; gap: .35rem; cursor: pointer;
  background: var(--ri-color); color: #fff; border: 0; border-radius: .7rem; padding: .5rem; font-weight: 600; font-size: .85rem;
}
.ri-btn-primario:hover { filter: brightness(1.08); }
.ri-btn-secundario {
  display: inline-flex; align-items: center; justify-content: center; width: 2.4rem; border-radius: .7rem;
  border: 1px solid hsl(var(--border)); color: hsl(var(--foreground));
}
.ri-anillo { position: relative; width: 56px; height: 56px; flex-shrink: 0; }
.ri-anillo svg { width: 56px; height: 56px; }
.ri-anillo-fondo { stroke: hsl(var(--muted)); }
.ri-anillo span { position: absolute; inset: 0; display: flex; align-items: center; justify-content: center; font-weight: 700; font-size: .9rem; }

/* ---------- plan de viaje ---------- */
.ri-plan { width: 100%; border-radius: 1.2rem; overflow: hidden; background: hsl(var(--card)); border: 1px solid hsl(var(--border)); }
.ri-plan-cabecera {
  display: flex; justify-content: space-between; align-items: flex-end; gap: 1rem; flex-wrap: wrap; padding: 1rem 1.2rem; color: #fff;
  background: linear-gradient(110deg, #1F5673, #2B6E8F 50%, #C8553D);
}
.ri-plan-cabecera small { font-size: .7rem; letter-spacing: .12em; opacity: .85; }
.ri-plan-cabecera h3 { margin: .1rem 0 0; font-size: 1.45rem; color: #fff; }
.ri-plan-cuerpo { padding: 1rem 1.2rem; display: flex; flex-direction: column; gap: .9rem; }
.ri-tramo { display: flex; align-items: center; gap: .8rem; }
.ri-tramo img { width: 96px; height: 64px; border-radius: .6rem; object-fit: cover; background: hsl(var(--muted)); }
.ri-num { width: 28px; height: 28px; border-radius: 99px; background: #C8553D; color: #fff; font-weight: 700; font-size: .8rem; display: flex; align-items: center; justify-content: center; flex-shrink: 0; }
.ri-reservar { display: inline-flex; align-items: center; gap: .2rem; font-size: .75rem; text-decoration: underline; margin-top: .15rem; }

/* ---------- métricas ---------- */
.ri-kpis { display: grid; grid-template-columns: repeat(auto-fill, minmax(170px, 1fr)); gap: .5rem; }
.ri-kpi { display: flex; align-items: center; gap: .6rem; background: hsl(var(--muted)); border-radius: .8rem; padding: .55rem .75rem; }
.ri-kpi small { display: block; font-size: .66rem; text-transform: uppercase; letter-spacing: .05em; color: hsl(var(--muted-foreground)); }
.ri-kpi b { font-size: .95rem; }

/* ---------- cercanías, aviso y respuestas sobre reseñas ---------- */
.ri-cerca { display: flex; flex-direction: column; gap: .15rem; font-size: .74rem; color: hsl(var(--muted-foreground)); }
.ri-cerca span { display: flex; align-items: center; gap: .3rem; min-width: 0; white-space: nowrap; }
.ri-cerca em { font-style: normal; overflow: hidden; text-overflow: ellipsis; flex: 1; min-width: 0; }
.ri-cerca svg { flex-shrink: 0; }
.ri-cerca b { color: hsl(var(--foreground)); font-weight: 600; flex-shrink: 0; }
.ri-aviso { font-size: .74rem; color: hsl(var(--muted-foreground)); display: flex; align-items: center; gap: .35rem; }
.ri-qa {
  width: 100%; display: flex; flex-direction: column; gap: .5rem; padding: .9rem 1rem; border-radius: 1rem;
  background: hsl(var(--card)); border: 1px solid hsl(var(--border)); border-left: 5px solid;
}
.ri-qa-cabecera { display: flex; align-items: center; gap: .6rem; flex-wrap: wrap; }
.ri-qa-veredicto { color: #fff; font-weight: 700; font-size: .72rem; letter-spacing: .06em; padding: .2rem .6rem; border-radius: 99px; }
.ri-cita small { font-style: normal; color: hsl(var(--muted-foreground)); }

In [ ]:
%%writefile public/theme.json
{
  "custom_fonts": [
    "https://fonts.googleapis.com/css2?family=Fraunces:opsz,wght@9..144,600;9..144,700&family=Inter:wght@400;500;600;700&display=swap"
  ],
  "variables": {
    "light": {
      "--font-sans": "'Inter', sans-serif",
      "--font-mono": "source-code-pro, Menlo, Monaco, Consolas, monospace",
      "--background": "38 45% 97%",
      "--foreground": "24 30% 14%",
      "--card": "0 0% 100%",
      "--card-foreground": "24 30% 14%",
      "--popover": "0 0% 100%",
      "--popover-foreground": "24 30% 14%",
      "--primary": "11 57% 51%",
      "--primary-foreground": "0 0% 100%",
      "--secondary": "36 55% 90%",
      "--secondary-foreground": "24 30% 14%",
      "--muted": "36 45% 92%",
      "--muted-foreground": "24 12% 42%",
      "--accent": "200 57% 29%",
      "--accent-foreground": "0 0% 100%",
      "--destructive": "0 72% 51%",
      "--destructive-foreground": "0 0% 100%",
      "--border": "33 30% 86%",
      "--input": "33 30% 86%",
      "--ring": "11 57% 51%",
      "--radius": "0.9rem",
      "--sidebar-background": "36 45% 94%",
      "--sidebar-foreground": "24 30% 14%",
      "--sidebar-primary": "11 57% 51%",
      "--sidebar-primary-foreground": "0 0% 100%",
      "--sidebar-accent": "36 45% 88%",
      "--sidebar-accent-foreground": "24 30% 14%",
      "--sidebar-border": "33 30% 86%",
      "--sidebar-ring": "11 57% 51%"
    },
    "dark": {
      "--font-sans": "'Inter', sans-serif",
      "--font-mono": "source-code-pro, Menlo, Monaco, Consolas, monospace",
      "--background": "205 35% 9%",
      "--foreground": "36 40% 94%",
      "--card": "205 32% 13%",
      "--card-foreground": "36 40% 94%",
      "--popover": "205 32% 13%",
      "--popover-foreground": "36 40% 94%",
      "--primary": "11 70% 60%",
      "--primary-foreground": "0 0% 100%",
      "--secondary": "205 25% 18%",
      "--secondary-foreground": "36 40% 94%",
      "--muted": "205 25% 17%",
      "--muted-foreground": "36 15% 70%",
      "--accent": "200 50% 40%",
      "--accent-foreground": "0 0% 100%",
      "--destructive": "0 62% 45%",
      "--destructive-foreground": "0 0% 100%",
      "--border": "205 20% 22%",
      "--input": "205 20% 22%",
      "--ring": "11 70% 60%",
      "--radius": "0.9rem",
      "--sidebar-background": "205 35% 7%",
      "--sidebar-foreground": "36 40% 94%",
      "--sidebar-primary": "11 70% 60%",
      "--sidebar-primary-foreground": "0 0% 100%",
      "--sidebar-accent": "205 25% 15%",
      "--sidebar-accent-foreground": "36 40% 94%",
      "--sidebar-border": "205 20% 20%",
      "--sidebar-ring": "11 70% 60%"
    }
  }
}

In [ ]:
%%writefile public/logo_light.svg
<svg xmlns="http://www.w3.org/2000/svg" width="260" height="56" viewBox="0 0 260 56">
  <circle cx="28" cy="28" r="24" fill="#1F5673"/>
  <path d="M8 34 Q18 26 28 34 T48 34 L48 42 Q38 50 28 50 Q14 50 8 42 Z" fill="#2B8FB5"/>
  <circle cx="34" cy="18" r="7" fill="#E9A23B"/>
  <path d="M14 36 L24 20 L31 30 L35 25 L44 36 Z" fill="#C8553D"/>
  <text x="62" y="36" font-family="Fraunces, Georgia, serif" font-size="26" font-weight="700" fill="#2A1E14">Ruta Ibérica</text>
</svg>

In [ ]:
%%writefile public/logo_dark.svg
<svg xmlns="http://www.w3.org/2000/svg" width="260" height="56" viewBox="0 0 260 56">
  <circle cx="28" cy="28" r="24" fill="#1F5673"/>
  <path d="M8 34 Q18 26 28 34 T48 34 L48 42 Q38 50 28 50 Q14 50 8 42 Z" fill="#2B8FB5"/>
  <circle cx="34" cy="18" r="7" fill="#E9A23B"/>
  <path d="M14 36 L24 20 L31 30 L35 25 L44 36 Z" fill="#C8553D"/>
  <text x="62" y="36" font-family="Fraunces, Georgia, serif" font-size="26" font-weight="700" fill="#F7EFE3">Ruta Ibérica</text>
</svg>

In [ ]:
%%writefile public/favicon.svg
<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 56 56"><circle cx="28" cy="28" r="26" fill="#1F5673"/><path d="M8 34 Q18 26 28 34 T48 34 L48 42 Q38 50 28 50 Q14 50 8 42 Z" fill="#2B8FB5"/><circle cx="34" cy="18" r="7" fill="#E9A23B"/><path d="M14 36 L24 20 L31 30 L35 25 L44 36 Z" fill="#C8553D"/></svg>

In [ ]:
%%writefile chainlit.md
# 🧭 Ruta Ibérica

**Asesor de viajes con IA para Barcelona, Madrid y Mallorca.**

## ¿Qué hace?
1. Interpreta tu pedido de viaje (destinos, fechas, huéspedes y presupuesto).
2. Busca alojamientos **disponibles en todas tus noches** y con **capacidad para todo el grupo**.
3. Te presenta **4 opciones por destino**:
   - 🧭 **Recomendada por nuestro modelo**, la de mayor puntaje Ruta Ibérica
   - ⭐ **Mejor rateada**, ajustada por cantidad de reseñas
   - 💸 **Más barata**
   - 💎 **Más cara**
4. Guarda tus elecciones y arma el resumen de la ruta con el costo total.

## ¿Cómo se calcula el puntaje (1 a 10)?
| Componente | Peso |
|---|---|
| Calidad percibida (rating de Airbnb con ajuste bayesiano) | 30 % |
| Análisis de reseñas con IA (limpieza, tranquilidad, anfitrión, fidelidad al anuncio, sentimiento) | 30 % |
| Relación precio/calidad frente a la mediana de su barrio | 20 % |
| Ubicación | 10 % |
| Ajuste de capacidad al tamaño del grupo | 10 % |

## Datos
- **Fuente:** [Inside Airbnb](https://insideairbnb.com/get-the-data/), relevamientos de junio de 2026 (Barcelona 24/06, Madrid 20/06, Mallorca 23/06).
- Catálogo curado de **1.500 alojamientos activos** (500 por destino) con reseñas recientes.
- Los precios son en euros por noche. La disponibilidad corresponde al último relevamiento y **puede haber cambiado**: confirmá siempre en Airbnb.

## Tecnología
LangChain (LCEL, Pydantic, RunnableParallel, batch, agente con tools y memoria) · Groq (modelos gpt-oss, free tier) · Chainlit.

In [ ]:
%%writefile .chainlit/config.toml
[project]
# List of environment variables to be provided by each user to use the app.
user_env = []

# Duration (in seconds) during which the session is saved when the connection is lost
session_timeout = 3600

# Duration (in seconds) of the user session expiry
user_session_timeout = 1296000  # 15 days

# Enable third parties caching (e.g., LangChain cache)
cache = false

# Whether to persist user environment variables (API keys) to the database
# Set to true to store user env vars in DB, false to exclude them for security
persist_user_env = false

# Whether to mask user environment variables (API keys) in the UI with password type
# Set to true to show API keys as ***, false to show them as plain text
mask_user_env = false

# Authorized origins
allow_origins = ["*"]

[features]
# Process and display HTML in messages. This can be a security risk (see https://stackoverflow.com/questions/19603097/why-is-it-dangerous-to-render-user-generated-html-or-javascript)
unsafe_allow_html = true

# Process and display mathematical expressions. This can clash with "$" characters in messages.
latex = false

# Enable rendering of user messages markdown
user_message_markdown = true

# Autoscroll new user messages at the top of the window
user_message_autoscroll = true

# Autoscroll new assistant messages
assistant_message_autoscroll = true

# Automatically tag threads with the current chat profile (if a chat profile is used)
auto_tag_thread = true

# Allow users to edit their own messages
edit_message = true

# Allow users to share threads (backend + UI). Requires an app-defined on_shared_thread_view callback.
allow_thread_sharing = false

# Enable favorite messages
favorites = false

[features.slack]
# Add emoji reaction when message is received (requires reactions:write OAuth scope)
reaction_on_message_received = false

# Authorize users to spontaneously upload files with messages
[features.spontaneous_file_upload]
    enabled = true
    # Define accepted file types using MIME types
    # Examples:
    # 1. For specific file types:
    #    accept = ["image/jpeg", "image/png", "application/pdf"]
    # 2. For all files of certain type:
    #    accept = ["image/*", "audio/*", "video/*"]
    # 3. For specific file extensions:
    #    accept = { "application/octet-stream" = [".xyz", ".pdb"] }
    # Note: Using "*/*" is not recommended as it may cause browser warnings
    accept = ["*/*"]
    max_files = 20
    max_size_mb = 500

[features.audio]
    # Enable audio features
    enabled = false
    # Sample rate of the audio
    sample_rate = 24000

[features.mcp]
    # Enable Model Context Protocol (MCP) features
    enabled = false

[features.mcp.sse]
    enabled = true

[features.mcp.streamable-http]
    enabled = true

[features.mcp.stdio]
    enabled = true
    # Only the executables in the allow list can be used for MCP stdio server.
    # Only need the base name of the executable, e.g. "npx", not "/usr/bin/npx".
    # Please don't comment this line for now, we need it to parse the executable name.
    allowed_executables = [ "npx", "uvx" ]

[UI]
# Name of the assistant.
name = "Ruta Ibérica"

# default_theme = "dark"

# Force a specific language for all users (e.g., "en-US", "he-IL", "fr-FR")
# If not set, the browser's language will be used
language = "es"

layout = "wide"

# default_sidebar_state = "open"  # Options: "open", "closed", "hidden"

# Chat settings display location: "message_composer" (default) or "sidebar" (header)
# chat_settings_location = "message_composer"

# Default state of chat settings sidebar when location is "sidebar"
# default_chat_settings_open = false

# Whether to prompt user confirmation on clicking 'New Chat'
confirm_new_chat = true

# Description of the assistant. This is used for HTML tags.
description = "Asesor de viajes con IA para Barcelona, Madrid y Mallorca basado en datos reales de Inside Airbnb"

# Chain of Thought (CoT) display mode. Can be "hidden", "tool_call" or "full".
cot = "tool_call"

# Specify a CSS file that can be used to customize the user interface.
# The CSS file can be served from the public directory or via an external link.
custom_css = "/public/custom.css"

# Specify additional attributes for a custom CSS file
# custom_css_attributes = "media=\"print\""

# Specify a JavaScript file that can be used to customize the user interface.
# The JavaScript file can be served from the public directory.
# custom_js = "/public/test.js"

# The style of alert boxes. Can be "classic" or "modern".
alert_style = "modern"

# Specify additional attributes for custom JS file
# custom_js_attributes = "async type = \"module\""

# Custom login page image, relative to public directory or external URL
# login_page_image = "/public/custom-background.jpg"

# Custom login page image filter (Tailwind internal filters, no dark/light variants)
# login_page_image_filter = "brightness-50 grayscale"
# login_page_image_dark_filter = "contrast-200 blur-sm"

# Specify a custom meta URL (used for meta tags like og:url)
# custom_meta_url = "https://github.com/Chainlit/chainlit"

# Specify a custom meta image url.
# custom_meta_image_url = "https://chainlit-cloud.s3.eu-west-3.amazonaws.com/logo/chainlit_banner.png"

# Load assistant logo directly from URL.
logo_file_url = ""

# Load assistant avatar image directly from URL.
default_avatar_file_url = "/public/favicon.svg"

# Avatar size in pixels (default: 20).
# avatar_size = 20

# Specify a custom build directory for the frontend.
# This can be used to customize the frontend code.
# Be careful: If this is a relative path, it should not start with a slash.
# custom_build = "./public/build"

# Specify optional one or more custom links in the header.
# [[UI.header_links]]
#     name = "Issues"
#     display_name = "Report Issue"
#     icon_url = "https://avatars.githubusercontent.com/u/128686189?s=200&v=4"
#     url = "https://github.com/Chainlit/chainlit/issues"
#     target = "_blank" (default)  # Optional: "_self", "_parent", "_top".

[meta]
generated_by = "2.11.0"

## 13 · Iniciar Chainlit y generar el enlace de acceso
- **En Colab:** la celda levanta Chainlit en segundo plano y crea un túnel público con `cloudflared` (no requiere cuenta).
  El enlace `https://....trycloudflare.com` aparece abajo.
- **En local:** ejecutar en una terminal `chainlit run app.py` y abrir http://localhost:8000

In [ ]:
import subprocess, re, time, os
if IN_COLAB:
    if not os.path.exists("cloudflared"):
        !wget -q -O cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
        !chmod +x cloudflared
    chainlit = subprocess.Popen(["chainlit", "run", "app.py", "--headless", "--port", "8000"],
                                stdout=open("chainlit.log", "w"), stderr=subprocess.STDOUT)
    time.sleep(8)
    tunel = subprocess.Popen(["./cloudflared", "tunnel", "--url", "http://localhost:8000", "--no-autoupdate"],
                             stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for linea in tunel.stdout:
        url = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", linea)
        if url:
            print("🧭 Ruta Ibérica disponible en:", url.group(0))
            break
else:
    print("Local: ejecutá  `chainlit run app.py`  en la carpeta del proyecto y abrí http://localhost:8000")